In [ ]:
# ============================================================
# PHASE 0 — Google Drive Status Check + Inventory
# Samsung PRISM — Kitchen Audio Dataset v2
#
# HOW TO RUN:
#   1. Open Google Colab (colab.research.google.com)
#   2. Create a new notebook
#   3. Copy this entire script into one cell
#   4. Run it
#
# WHAT IT DOES:
#   Part A — Checks your Drive structure (what exists, what's missing)
#   Part B — Counts usable files per class per source
#   Part C — Flags problems (too short, wrong format, corrupt)
#   Part D — Tells you exactly how many clips you can generate
#   Part E — Saves inventory.csv + pipeline_report.xlsx to Drive
# ============================================================
# ── Cell 1: Mount Drive + Install dependencies ────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import uuid
import warnings
import traceback
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION — edit these paths to match your Drive
# ============================================================

# Root of your raw source files
RAW_SOURCES_ROOT = "/content/drive/MyDrive/kitpri/raw_sources"

# Where output files (inventory.csv, Excel) will be saved
OUTPUT_DIR = "/content/drive/MyDrive/kitpri/kitpri_v2/metadata"

# Minimum duration (seconds) for a source file to be considered usable
MIN_DURATION_SEC = 1.0

# Maximum times one source file can be reused across clips
MAX_REUSE = 3

# Expected folder structure:
#   RAW_SOURCES_ROOT/
#       freesound/frying/*.wav
#       freesound/chopping/*.wav
#       fsd50k/boiling/*.wav
#       musan/speech/*.wav
#       esc50/street_noise/*.wav
#       ...

# ── Define your classes ──────────────────────────────────────

COOKING_FG_CLASSES = [
    "frying", "chopping", "boiling", "blender", "sizzling",
    "knife_scraping", "microwave_beep", "pressure_cooker",
    "water_running", "mortar_pestle",
]

NONCOOKING_FG_CLASSES = [
    "speech", "TV_audio", "music", "keyboard_typing", "dog_barking",
    "phone_ringing", "door_knock", "baby_crying", "vacuum_cleaner", "footsteps",
]

BACKGROUND_CLASSES = [
    "kitchen_ambience", "living_room_tone", "street_noise",
    "office_noise", "rain", "HVAC", "crowd_distant", "silence",
]

ALL_EXPECTED_CLASSES = {
    "cooking_fg":     COOKING_FG_CLASSES,
    "noncooking_fg":  NONCOOKING_FG_CLASSES,
    "background":     BACKGROUND_CLASSES,
}

KNOWN_SOURCES = ["freesound", "fsd50k", "musan", "esc50", "audioset", "ambience"]

AUDIO_EXTENSIONS = {".wav", ".mp3", ".flac", ".ogg", ".m4a"}

# ============================================================
# PART A — DRIVE STRUCTURE CHECK
# ============================================================

def check_drive_structure(raw_root: str) -> dict:
    """
    Scans RAW_SOURCES_ROOT and reports:
    - What source dataset folders exist
    - What class folders exist inside each
    - What's expected but missing
    """
    print("\n" + "="*60)
    print("  PART A — Drive Structure Check")
    print("="*60)

    report = {
        "root_exists":       False,
        "found_sources":     [],
        "missing_sources":   [],
        "class_structure":   {},   # source → list of class folders found
        "missing_classes":   {},   # source → list of expected classes missing
        "unexpected_folders":[],
    }

    if not os.path.exists(raw_root):
        print(f"\n❌ RAW_SOURCES_ROOT not found: {raw_root}")
        print("   → Create this folder in Drive and add your source files.")
        print("   → Expected structure:")
        print("       kitpri/raw_sources/freesound/frying/")
        print("       kitpri/raw_sources/fsd50k/boiling/")
        print("       kitpri/raw_sources/musan/speech/")
        print("       kitpri/raw_sources/esc50/street_noise/")
        return report

    report["root_exists"] = True
    print(f"\n✅ Root found: {raw_root}")

    # Scan top-level (source dataset folders)
    top_level = [d for d in os.listdir(raw_root)
                 if os.path.isdir(os.path.join(raw_root, d))]

    print(f"\n  Source dataset folders found: {top_level}")

    for src in top_level:
        src_path = os.path.join(raw_root, src)
        class_folders = [d for d in os.listdir(src_path)
                         if os.path.isdir(os.path.join(src_path, d))]
        report["class_structure"][src] = class_folders
        report["found_sources"].append(src)

        if src.lower() not in KNOWN_SOURCES:
            report["unexpected_folders"].append(src)

    # Check for missing sources
    for src in KNOWN_SOURCES:
        if src not in [s.lower() for s in top_level]:
            report["missing_sources"].append(src)

    # Print structure
    all_classes_flat = (COOKING_FG_CLASSES + NONCOOKING_FG_CLASSES + BACKGROUND_CLASSES)

    print(f"\n  Detailed structure:")
    for src, classes in report["class_structure"].items():
        print(f"\n  📁 {src}/")
        for cls in sorted(classes):
            cls_path = os.path.join(raw_root, src, cls)
            n_files  = len([f for f in os.listdir(cls_path)
                            if Path(f).suffix.lower() in AUDIO_EXTENSIONS])
            status   = "✅" if cls in all_classes_flat else "⚠️  (unexpected)"
            print(f"      ├── {cls}/  [{n_files} audio files]  {status}")

        # Check which expected classes are missing from this source
        missing = [c for c in all_classes_flat if c not in classes]
        if missing:
            report["missing_classes"][src] = missing

    # Summary
    print(f"\n  Missing source folders (no files yet):")
    for src in report["missing_sources"]:
        print(f"      ❌ {src}/  — not found in Drive")

    if report["unexpected_folders"]:
        print(f"\n  ⚠️  Unexpected folders (not in known sources list):")
        for f in report["unexpected_folders"]:
            print(f"      {f}/  — check if this is intentional")

    return report


# ============================================================
# PART B — FILE INVENTORY (per source per class)
# ============================================================

def generate_source_file_id() -> str:
    return "SRC_" + uuid.uuid4().hex[:6].upper()


def inspect_audio_file(filepath: str) -> dict:
    """
    Loads file metadata without fully decoding.
    Returns duration, sample_rate, channels, is_usable, error_message.
    """
    result = {
        "duration_sec":  None,
        "sample_rate":   None,
        "channels":      None,
        "file_size_kb":  round(os.path.getsize(filepath) / 1024, 1),
        "is_usable":     False,
        "error_message": "",
    }
    try:
        info = sf.info(filepath)
        result["duration_sec"] = round(info.duration, 2)
        result["sample_rate"]  = info.samplerate
        result["channels"]     = info.channels
        result["is_usable"]    = info.duration >= MIN_DURATION_SEC
        if not result["is_usable"]:
            result["error_message"] = f"Too short ({info.duration:.2f}s < {MIN_DURATION_SEC}s)"
    except Exception:
        try:
            # Fallback: try librosa
            y, sr = librosa.load(filepath, sr=None, mono=False)
            dur   = librosa.get_duration(y=y, sr=sr)
            result["duration_sec"] = round(dur, 2)
            result["sample_rate"]  = sr
            result["channels"]     = 1 if y.ndim == 1 else y.shape[0]
            result["is_usable"]    = dur >= MIN_DURATION_SEC
            if not result["is_usable"]:
                result["error_message"] = f"Too short ({dur:.2f}s)"
        except Exception as e:
            result["error_message"] = f"Cannot read: {str(e)[:80]}"
            result["is_usable"]     = False

    return result


def build_inventory(raw_root: str) -> pd.DataFrame:
    """
    Walks all source files and builds a row per file.
    """
    print("\n" + "="*60)
    print("  PART B — File Inventory")
    print("="*60)

    all_classes_flat = set(COOKING_FG_CLASSES + NONCOOKING_FG_CLASSES + BACKGROUND_CLASSES)

    def get_role_category(cls: str) -> str:
        if cls in COOKING_FG_CLASSES:    return "cooking_fg"
        if cls in NONCOOKING_FG_CLASSES: return "noncooking_fg"
        if cls in BACKGROUND_CLASSES:    return "background"
        return "unknown"

    rows    = []
    total   = 0
    usable  = 0
    skipped = 0

    for src_dataset in sorted(os.listdir(raw_root)):
        src_path = os.path.join(raw_root, src_dataset)
        if not os.path.isdir(src_path):
            continue

        for audio_class in sorted(os.listdir(src_path)):
            cls_path = os.path.join(src_path, audio_class)
            if not os.path.isdir(cls_path):
                continue

            files = [f for f in os.listdir(cls_path)
                     if Path(f).suffix.lower() in AUDIO_EXTENSIONS]

            print(f"  Scanning {src_dataset}/{audio_class}/ — {len(files)} files...", end="")

            for fname in files:
                total += 1
                fpath  = os.path.join(cls_path, fname)
                info   = inspect_audio_file(fpath)

                if info["is_usable"]:
                    usable += 1
                else:
                    skipped += 1

                rows.append({
                    "source_file_id":  generate_source_file_id(),
                    "file_name":       fname,
                    "file_path":       fpath,
                    "source_dataset":  src_dataset,
                    "audio_class":     audio_class,
                    "role_category":   get_role_category(audio_class),
                    "duration_sec":    info["duration_sec"],
                    "sample_rate":     info["sample_rate"],
                    "channels":        info["channels"],
                    "file_size_kb":    info["file_size_kb"],
                    "is_usable":       int(info["is_usable"]),
                    "times_used":      0,
                    "error_message":   info["error_message"],
                    "scanned_at":      datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                })

            usable_here = sum(1 for r in rows[-len(files):]
                              if r["is_usable"] == 1)
            print(f"  {usable_here} usable")

    df = pd.DataFrame(rows)
    print(f"\n  Total files scanned : {total}")
    print(f"  Usable              : {usable}")
    print(f"  Skipped (too short / corrupt) : {skipped}")
    return df


# ============================================================
# PART C — CLASS COVERAGE REPORT
# ============================================================

def class_coverage_report(inventory_df: pd.DataFrame) -> pd.DataFrame:
    """
    Per-class summary: how many usable files, from which sources,
    total available duration, and max clips generatable.
    """
    print("\n" + "="*60)
    print("  PART C — Class Coverage Report")
    print("="*60)

    usable = inventory_df[inventory_df["is_usable"] == 1]

    summary_rows = []
    for role, classes in ALL_EXPECTED_CLASSES.items():
        for cls in classes:
            cls_df        = usable[usable["audio_class"] == cls]
            n_files       = len(cls_df)
            sources       = ", ".join(sorted(cls_df["source_dataset"].unique())) if n_files > 0 else "MISSING"
            total_dur     = round(cls_df["duration_sec"].sum(), 1) if n_files > 0 else 0
            avg_dur       = round(cls_df["duration_sec"].mean(), 1) if n_files > 0 else 0
            max_clips     = n_files * MAX_REUSE
            has_enough    = "✅" if n_files >= 10 else ("⚠️" if n_files > 0 else "❌ MISSING")

            summary_rows.append({
                "role_category":    role,
                "audio_class":      cls,
                "usable_files":     n_files,
                "sources":          sources,
                "total_duration_s": total_dur,
                "avg_duration_s":   avg_dur,
                "max_clips":        max_clips,
                "status":           has_enough,
            })

    df = pd.DataFrame(summary_rows)

    # Print by category
    for role in ["cooking_fg", "noncooking_fg", "background"]:
        print(f"\n  {role.upper()}")
        print(f"  {'Class':<22} {'Files':>6} {'Max Clips':>10} {'Avg Dur':>8} {'Sources':<30} {'Status'}")
        print(f"  {'-'*90}")
        sub = df[df["role_category"] == role]
        for _, row in sub.iterrows():
            print(f"  {row['audio_class']:<22} {row['usable_files']:>6} "
                  f"{row['max_clips']:>10} {row['avg_duration_s']:>7}s "
                  f"  {row['sources']:<30} {row['status']}")

    return df


# ============================================================
# PART D — DATASET SIZE RECOMMENDATION
# ============================================================

def recommend_dataset_size(coverage_df: pd.DataFrame) -> dict:
    """
    Derives the maximum safe dataset size from inventory.
    Logic:
        - cooking clips limited by weakest cooking FG class
        - non-cooking clips limited by weakest non-cooking FG class
        - final dataset = min(both) × 2, balanced
    """
    print("\n" + "="*60)
    print("  PART D — Dataset Size Recommendation")
    print("="*60)

    cooking_fg    = coverage_df[coverage_df["role_category"] == "cooking_fg"]
    noncooking_fg = coverage_df[coverage_df["role_category"] == "noncooking_fg"]
    background    = coverage_df[coverage_df["role_category"] == "background"]

    min_cooking    = cooking_fg["usable_files"].min()
    min_noncooking = noncooking_fg["usable_files"].min()
    min_bg         = background["usable_files"].min()

    weak_cooking_class    = cooking_fg.loc[cooking_fg["usable_files"].idxmin(), "audio_class"]
    weak_noncooking_class = noncooking_fg.loc[noncooking_fg["usable_files"].idxmin(), "audio_class"]
    weak_bg_class         = background.loc[background["usable_files"].idxmin(), "audio_class"]

    # Each clip uses 2 FG stems from the same category
    # So max cooking clips = min_cooking_class_files * MAX_REUSE (conservative)
    max_cooking_clips    = min_cooking * MAX_REUSE
    max_noncooking_clips = min_noncooking * MAX_REUSE

    # Balance: equal cooking and non-cooking
    max_balanced = min(max_cooking_clips, max_noncooking_clips)

    # Recommended sizes
    recommended = {
        "conservative": min(max_balanced, 2000),    # safe floor
        "standard":     min(max_balanced, 5000),    # good for training
        "full":         max_balanced * 2,           # use everything available
    }

    print(f"\n  Weakest cooking class    : {weak_cooking_class} ({min_cooking} files)")
    print(f"  Weakest non-cooking class: {weak_noncooking_class} ({min_noncooking} files)")
    print(f"  Weakest background class : {weak_bg_class} ({min_bg} files)")

    print(f"\n  ┌─────────────────────────────────────────────────┐")
    print(f"  │  Max cooking clips (3× reuse limit)  : {max_cooking_clips:>6}   │")
    print(f"  │  Max non-cooking clips               : {max_noncooking_clips:>6}   │")
    print(f"  ├─────────────────────────────────────────────────┤")
    print(f"  │  CONSERVATIVE (safe)    : {recommended['conservative']:>6} total clips  │")
    print(f"  │  STANDARD (recommended) : {recommended['standard']:>6} total clips  │")
    print(f"  │  FULL (use everything)  : {recommended['full']:>6} total clips  │")
    print(f"  └─────────────────────────────────────────────────┘")

    print(f"\n  ℹ️  Recommendation: start with STANDARD ({recommended['standard']} clips)")
    print(f"     If any class has < 10 files, download more before generating.")

    # Flag classes needing more files
    print(f"\n  Classes needing attention (< 10 usable files):")
    needs_work = coverage_df[coverage_df["usable_files"] < 10]
    if needs_work.empty:
        print(f"      ✅ All classes have sufficient files")
    else:
        for _, row in needs_work.iterrows():
            needed = max(0, 10 - row["usable_files"])
            print(f"      ❌ {row['audio_class']} ({row['role_category']}): "
                  f"{row['usable_files']} files — need {needed} more")
            if row["audio_class"] in COOKING_FG_CLASSES:
                print(f"         → Download from: Freesound.org (search '{row['audio_class']} sound')")
            elif row["audio_class"] in NONCOOKING_FG_CLASSES:
                print(f"         → Download from: MUSAN or ESC-50")
            else:
                print(f"         → Download from: ESC-50 or MUSAN noise set")

    return recommended


# ============================================================
# PART E — EXPORT TO EXCEL + CSV
# ============================================================

HEADER_FILL  = PatternFill("solid", start_color="1F4E79")
GREEN_FILL   = PatternFill("solid", start_color="C6EFCE")
YELLOW_FILL  = PatternFill("solid", start_color="FFEB9C")
RED_FILL     = PatternFill("solid", start_color="FFC7CE")
ALT_FILL     = PatternFill("solid", start_color="D6E4F0")
WHITE_FONT   = Font(name="Arial", bold=True, color="FFFFFF", size=10)
BODY_FONT    = Font(name="Arial", size=9)
BOLD_FONT    = Font(name="Arial", bold=True, size=9)
CENTER       = Alignment(horizontal="center", vertical="center", wrap_text=True)
LEFT         = Alignment(horizontal="left", vertical="center")


def _border():
    s = Side(style="thin")
    return Border(left=s, right=s, top=s, bottom=s)


def _write_sheet(ws, df: pd.DataFrame, status_col: str = None):
    B = _border()
    for c, name in enumerate(df.columns, 1):
        cell = ws.cell(row=1, column=c, value=name)
        cell.fill = HEADER_FILL; cell.font = WHITE_FONT
        cell.alignment = CENTER; cell.border = B

    status_idx = list(df.columns).index(status_col) + 1 if status_col and status_col in df.columns else None

    for r, row in enumerate(df.itertuples(index=False), 2):
        status_val = ws.cell(row=r, column=status_idx).value if status_idx else None
        if status_val == "✅":   fill = GREEN_FILL
        elif status_val == "⚠️": fill = YELLOW_FILL
        elif status_val == "❌ MISSING": fill = RED_FILL
        else: fill = ALT_FILL if r % 2 == 0 else None

        for c, val in enumerate(row, 1):
            cell = ws.cell(row=r, column=c, value=val)
            cell.font = BODY_FONT; cell.alignment = LEFT; cell.border = B
            if fill: cell.fill = fill

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    for col in ws.columns:
        w = max((len(str(c.value or "")) for c in col), default=8)
        ws.column_dimensions[get_column_letter(col[0].column)].width = min(w + 2, 45)


def export_inventory(
    inventory_df:  pd.DataFrame,
    coverage_df:   pd.DataFrame,
    recommended:   dict,
    output_dir:    str,
):
    os.makedirs(output_dir, exist_ok=True)

    # Save CSVs
    inv_csv = os.path.join(output_dir, "inventory.csv")
    cov_csv = os.path.join(output_dir, "coverage_summary.csv")
    inventory_df.to_csv(inv_csv, index=False)
    coverage_df.to_csv(cov_csv, index=False)
    print(f"\n  ✅ inventory.csv saved      → {inv_csv}")
    print(f"  ✅ coverage_summary.csv saved → {cov_csv}")

    # Build Excel
    wb = Workbook()

    # Sheet 1: Source Inventory (all raw files)
    ws1 = wb.active; ws1.title = "Source Inventory"
    _write_sheet(ws1, inventory_df)

    # Sheet 2: Class Coverage
    ws2 = wb.create_sheet("Class Coverage")
    _write_sheet(ws2, coverage_df, status_col="status")

    # Sheet 3: Dataset Size Options
    ws3 = wb.create_sheet("Dataset Size Options")
    size_data = pd.DataFrame([
        {"option": "Conservative", "total_clips": recommended["conservative"],
         "cooking": recommended["conservative"]//2,
         "non_cooking": recommended["conservative"]//2,
         "note": "Safe — use if some classes have few files"},
        {"option": "Standard ✅ Recommended", "total_clips": recommended["standard"],
         "cooking": recommended["standard"]//2,
         "non_cooking": recommended["standard"]//2,
         "note": "Good balance of size vs training time"},
        {"option": "Full", "total_clips": recommended["full"],
         "cooking": recommended["full"]//2,
         "non_cooking": recommended["full"]//2,
         "note": "Use everything — best for final submission"},
    ])
    _write_sheet(ws3, size_data)

    # Sheet 4: Missing Classes Action Plan
    ws4 = wb.create_sheet("Action Plan")
    missing = coverage_df[coverage_df["usable_files"] < 10].copy()
    if missing.empty:
        missing = pd.DataFrame([{"message": "All classes have sufficient files ✅"}])
    else:
        missing["files_needed"] = (10 - missing["usable_files"]).clip(lower=0)
        missing["download_from"] = missing["audio_class"].apply(
            lambda c: "Freesound.org" if c in COOKING_FG_CLASSES
            else ("MUSAN / ESC-50" if c in NONCOOKING_FG_CLASSES
                  else "ESC-50 / MUSAN noise")
        )
        missing["search_query"] = missing["audio_class"].apply(
            lambda c: f'"{c.replace("_", " ")} sound" site:freesound.org'
        )
    _write_sheet(ws4, missing)

    # Sheet 5: Placeholder for future phases
    for sheet_name in ["Generation Plan", "Mixed Clips Metadata",
                       "Train Split", "Val Split", "Test Split",
                       "Training Log", "Prediction Results", "Error Analysis"]:
        ws = wb.create_sheet(sheet_name)
        ws["A1"] = f"← Will be populated in a later phase"
        ws["A1"].font = Font(name="Arial", italic=True, color="888888")

    xlsx_path = os.path.join(output_dir, "pipeline_report.xlsx")
    wb.save(xlsx_path)
    print(f"  ✅ pipeline_report.xlsx saved → {xlsx_path}")

    return xlsx_path


# ============================================================
# MAIN — run everything
# ============================================================

def main():
    print("\n" + "="*60)
    print("  PHASE 0 — Drive Inventory & Status Check")
    print("  Samsung PRISM — Kitchen Audio Dataset v2")
    print(f"  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*60)

    # Part A: structure check
    structure = check_drive_structure(RAW_SOURCES_ROOT)

    if not structure["root_exists"]:
        print("\n⛔ Cannot proceed — fix Drive structure first.")
        print(f"   Create: {RAW_SOURCES_ROOT}")
        print("   Then add your source audio files and re-run.")
        return

    # Part B: inventory
    inventory_df = build_inventory(RAW_SOURCES_ROOT)

    if inventory_df.empty:
        print("\n⛔ No audio files found. Add source files to Drive and re-run.")
        return

    # Part C: class coverage
    coverage_df = class_coverage_report(inventory_df)

    # Part D: size recommendation
    recommended = recommend_dataset_size(coverage_df)

    # Part E: export
    print("\n" + "="*60)
    print("  PART E — Saving Output Files")
    print("="*60)
    export_inventory(inventory_df, coverage_df, recommended, OUTPUT_DIR)

    # Final summary
    print("\n" + "="*60)
    print("  PHASE 0 COMPLETE")
    print("="*60)
    total_usable = inventory_df[inventory_df["is_usable"] == 1].shape[0]
    total_all    = len(inventory_df)
    print(f"\n  Total source files found : {total_all}")
    print(f"  Usable files             : {total_usable}")
    print(f"  Unusable (too short etc) : {total_all - total_usable}")
    print(f"\n  Recommended dataset size : {recommended['standard']} clips")
    print(f"  Output saved to          : {OUTPUT_DIR}")
    print(f"\n  NEXT STEP → Phase 1: Generation Plan")
    print(f"  (only after all ❌ MISSING classes are resolved)")
    print("="*60)


main()

In [ ]:
#fixng thr phase 0 file

import os
from pathlib import Path

ROOT = "/content/drive/MyDrive/kitpri/raw_sources"
AUDIO_EXT = {".wav", ".mp3", ".flac", ".ogg", ".m4a"}

print("Scanning actual structure...\n")
for top in sorted(os.listdir(ROOT)):              # foreground, background
    top_path = os.path.join(ROOT, top)
    if not os.path.isdir(top_path): continue
    for mid in sorted(os.listdir(top_path)):      # cooking, noncooking OR class name
        mid_path = os.path.join(top_path, mid)
        if not os.path.isdir(mid_path): continue
        sub = os.listdir(mid_path)
        has_subdirs = any(os.path.isdir(os.path.join(mid_path, s)) for s in sub)
        if has_subdirs:
            for cls in sorted(sub):              # actual class folders
                cls_path = os.path.join(mid_path, cls)
                if not os.path.isdir(cls_path): continue
                n = len([f for f in os.listdir(cls_path)
                         if Path(f).suffix.lower() in AUDIO_EXT])
                print(f"  {top}/{mid}/{cls:<25} {n:>4} files")
        else:
            n = len([f for f in sub if Path(f).suffix.lower() in AUDIO_EXT])
            print(f"  {top}/{mid:<35} {n:>4} files")

In [ ]:
# ============================================================
# PHASE 0 — Drive Inventory & Status Check (v2 — fixed paths)
# Samsung PRISM — Kitchen Audio Dataset v2
#
# Folder structure expected:
#   raw_sources/
#     foreground/
#       cooking/    frying/ boiling/ chopping/ ...
#       noncooking/ speech/ TV_audio/ music/ ...
#     background/
#       fan_ac/ musan_noise/ rain/ room_ambience/ street_noise/ ...
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os, uuid, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================

RAW_SOURCES_ROOT = "/content/drive/MyDrive/kitpri/raw_sources"
OUTPUT_DIR       = "/content/drive/MyDrive/kitpri/kitpri_v2/metadata"
MIN_DURATION_SEC = 1.0
MAX_REUSE        = 3
AUDIO_EXT        = {".wav", ".mp3", ".flac", ".ogg", ".m4a"}

# ── Class definitions — matched to YOUR actual folder names ──

COOKING_FG_CLASSES = [
    "frying", "chopping", "boiling", "blender", "stove",
    "dishes", "microwave", "pressure_cooker", "water_tap", "stirring",
]

NONCOOKING_FG_CLASSES = [
    "speech", "TV_audio", "music", "keyboard_typing", "dog_barking",
    "phone_ringing", "door_knock", "vacuum_cleaner", "footsteps",
]

BACKGROUND_CLASSES = [
    "room_ambience", "fan_ac", "musan_noise",
    "street_noise", "rain", "noncooking_other",
]

ALL_CLASSES = {
    "cooking_fg":    COOKING_FG_CLASSES,
    "noncooking_fg": NONCOOKING_FG_CLASSES,
    "background":    BACKGROUND_CLASSES,
}

# ── Path map: class_name → actual path in raw_sources ────────
# This handles the two-level FG structure and flat BG structure

def build_path_map(root: str) -> dict:
    """
    Returns {class_name: full_path} for every class folder found.
    Handles:
      foreground/cooking/<class>/
      foreground/noncooking/<class>/
      background/<class>/
    """
    path_map = {}

    fg_root = os.path.join(root, "foreground")
    bg_root = os.path.join(root, "background")

    # Foreground: two levels deep
    if os.path.exists(fg_root):
        for category in os.listdir(fg_root):            # cooking, noncooking
            cat_path = os.path.join(fg_root, category)
            if not os.path.isdir(cat_path):
                continue
            for cls in os.listdir(cat_path):            # frying, speech, etc.
                cls_path = os.path.join(cat_path, cls)
                if os.path.isdir(cls_path):
                    path_map[cls] = cls_path

    # Background: one level deep
    if os.path.exists(bg_root):
        for cls in os.listdir(bg_root):
            cls_path = os.path.join(bg_root, cls)
            if os.path.isdir(cls_path):
                path_map[cls] = cls_path

    return path_map


def get_role(cls: str) -> str:
    if cls in COOKING_FG_CLASSES:    return "cooking_fg"
    if cls in NONCOOKING_FG_CLASSES: return "noncooking_fg"
    if cls in BACKGROUND_CLASSES:    return "background"
    return "unknown"


# ============================================================
# PART A — STRUCTURE CHECK
# ============================================================

def check_structure(root: str) -> bool:
    print("\n" + "="*60)
    print("  PART A — Drive Structure Check")
    print("="*60)

    if not os.path.exists(root):
        print(f"\n❌ Not found: {root}")
        return False

    print(f"\n✅ Root found: {root}")
    path_map = build_path_map(root)

    print(f"\n  Folders found ({len(path_map)} classes):\n")
    for cls, path in sorted(path_map.items()):
        n    = len([f for f in os.listdir(path)
                    if Path(f).suffix.lower() in AUDIO_EXT])
        role = get_role(cls)
        flag = "✅" if n >= 20 else ("⚠️ " if n > 0 else "❌")
        rel  = path.replace(root + "/", "")
        print(f"  {flag} {rel:<45} {n:>4} files  [{role}]")

    # Check for expected classes with no folder
    all_expected = COOKING_FG_CLASSES + NONCOOKING_FG_CLASSES + BACKGROUND_CLASSES
    missing      = [c for c in all_expected if c not in path_map]
    if missing:
        print(f"\n  ⚠️  Expected classes not found as folders:")
        for m in missing:
            print(f"      ❌ {m}  ({get_role(m)})")

    return True


# ============================================================
# PART B — FILE INVENTORY
# ============================================================

def inspect_file(filepath: str) -> dict:
    result = {
        "duration_sec": None, "sample_rate": None,
        "channels": None,
        "file_size_kb": round(os.path.getsize(filepath) / 1024, 1),
        "is_usable": False, "error_message": "",
    }
    try:
        info = sf.info(filepath)
        result.update({
            "duration_sec": round(info.duration, 2),
            "sample_rate":  info.samplerate,
            "channels":     info.channels,
            "is_usable":    info.duration >= MIN_DURATION_SEC,
        })
        if not result["is_usable"]:
            result["error_message"] = f"Too short ({info.duration:.2f}s)"
    except Exception:
        try:
            y, sr = librosa.load(filepath, sr=None, mono=False)
            dur   = librosa.get_duration(y=y, sr=sr)
            result.update({
                "duration_sec": round(dur, 2),
                "sample_rate":  sr,
                "channels":     1 if y.ndim == 1 else y.shape[0],
                "is_usable":    dur >= MIN_DURATION_SEC,
            })
        except Exception as e:
            result["error_message"] = f"Cannot read: {str(e)[:80]}"
    return result


def build_inventory(root: str) -> pd.DataFrame:
    print("\n" + "="*60)
    print("  PART B — File Inventory")
    print("="*60)

    path_map = build_path_map(root)
    rows     = []
    total = usable = skipped = 0

    for cls, cls_path in sorted(path_map.items()):
        files = [f for f in os.listdir(cls_path)
                 if Path(f).suffix.lower() in AUDIO_EXT]
        role  = get_role(cls)

        # Determine source_dataset from path
        rel   = cls_path.replace(root + "/", "")
        parts = rel.split("/")
        src   = "/".join(parts[:-1])   # e.g. "foreground/cooking"

        print(f"  Scanning {rel:<40} {len(files):>4} files ... ", end="")

        n_usable = 0
        for fname in files:
            total += 1
            info   = inspect_file(os.path.join(cls_path, fname))
            if info["is_usable"]:
                usable += 1; n_usable += 1
            else:
                skipped += 1

            rows.append({
                "source_file_id": "SRC_" + uuid.uuid4().hex[:6].upper(),
                "file_name":      fname,
                "file_path":      os.path.join(cls_path, fname),
                "source_dataset": src,
                "audio_class":    cls,
                "role_category":  role,
                "duration_sec":   info["duration_sec"],
                "sample_rate":    info["sample_rate"],
                "channels":       info["channels"],
                "file_size_kb":   info["file_size_kb"],
                "is_usable":      int(info["is_usable"]),
                "times_used":     0,
                "error_message":  info["error_message"],
                "scanned_at":     datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            })

        print(f"{n_usable} usable")

    print(f"\n  Total : {total}  |  Usable : {usable}  |  Skipped : {skipped}")
    return pd.DataFrame(rows)


# ============================================================
# PART C — CLASS COVERAGE
# ============================================================

def class_coverage(inventory_df: pd.DataFrame) -> pd.DataFrame:
    print("\n" + "="*60)
    print("  PART C — Class Coverage Report")
    print("="*60)

    usable = inventory_df[inventory_df["is_usable"] == 1]
    rows   = []

    for role, classes in ALL_CLASSES.items():
        print(f"\n  {role.upper()}")
        print(f"  {'Class':<22} {'Files':>6} {'MaxClips':>9} {'AvgDur':>7}  Status")
        print(f"  {'-'*65}")

        for cls in classes:
            df        = usable[usable["audio_class"] == cls]
            n         = len(df)
            avg_dur   = round(df["duration_sec"].mean(), 1) if n > 0 else 0.0
            max_clips = n * MAX_REUSE
            status    = "✅" if n >= 20 else ("⚠️ " if n > 0 else "❌ MISSING")
            sources   = ", ".join(sorted(df["source_dataset"].unique())) if n > 0 else "—"

            print(f"  {cls:<22} {n:>6} {max_clips:>9} {avg_dur:>6}s  {status}")
            rows.append({
                "role_category":    role,
                "audio_class":      cls,
                "usable_files":     n,
                "sources":          sources,
                "avg_duration_s":   avg_dur,
                "max_clips":        max_clips,
                "status":           status.strip(),
            })

    return pd.DataFrame(rows)


# ============================================================
# PART D — DATASET SIZE RECOMMENDATION
# ============================================================

def recommend_size(coverage_df: pd.DataFrame) -> dict:
    print("\n" + "="*60)
    print("  PART D — Dataset Size Recommendation")
    print("="*60)

    ckg = coverage_df[coverage_df["role_category"] == "cooking_fg"]
    ncg = coverage_df[coverage_df["role_category"] == "noncooking_fg"]
    bg  = coverage_df[coverage_df["role_category"] == "background"]

    min_c  = ckg["usable_files"].min()
    min_nc = ncg["usable_files"].min()
    min_bg = bg["usable_files"].min()

    weak_c  = ckg.loc[ckg["usable_files"].idxmin(), "audio_class"]
    weak_nc = ncg.loc[ncg["usable_files"].idxmin(), "audio_class"]
    weak_bg = bg.loc[bg["usable_files"].idxmin(), "audio_class"]

    max_cooking    = min_c  * MAX_REUSE
    max_noncooking = min_nc * MAX_REUSE
    max_balanced   = min(max_cooking, max_noncooking)

    recommended = {
        "conservative": min(max_balanced, 2000),
        "standard":     min(max_balanced, 5000),
        "full":         max_balanced * 2,
    }

    print(f"\n  Weakest cooking class    : {weak_c}  ({min_c} files → {max_cooking} max clips)")
    print(f"  Weakest non-cooking class: {weak_nc} ({min_nc} files → {max_noncooking} max clips)")
    print(f"  Weakest background class : {weak_bg} ({min_bg} files)")

    print(f"\n  ┌─────────────────────────────────────────────┐")
    print(f"  │  CONSERVATIVE  : {recommended['conservative']:>5} clips (safe floor)    │")
    print(f"  │  STANDARD  ✅  : {recommended['standard']:>5} clips (recommended)   │")
    print(f"  │  FULL          : {recommended['full']:>5} clips (use everything)  │")
    print(f"  └─────────────────────────────────────────────┘")

    # Flag anything needing attention
    needs_work = coverage_df[coverage_df["usable_files"] < 20]
    if not needs_work.empty:
        print(f"\n  Classes needing more files (< 20):")
        for _, row in needs_work.iterrows():
            needed = max(0, 20 - row["usable_files"])
            print(f"    ⚠️  {row['audio_class']:<22} {row['usable_files']:>3} files "
                  f"— ideally {needed} more")
    else:
        print(f"\n  ✅ All classes have 20+ files")

    return recommended


# ============================================================
# PART E — EXCEL EXPORT
# ============================================================

HDR  = PatternFill("solid", start_color="1F4E79")
GRN  = PatternFill("solid", start_color="C6EFCE")
YLW  = PatternFill("solid", start_color="FFEB9C")
RED  = PatternFill("solid", start_color="FFC7CE")
ALT  = PatternFill("solid", start_color="D6E4F0")
WFNT = Font(name="Arial", bold=True, color="FFFFFF", size=10)
BFNT = Font(name="Arial", size=9)
CTR  = Alignment(horizontal="center", vertical="center", wrap_text=True)
LFT  = Alignment(horizontal="left",   vertical="center")

def _bdr():
    s = Side(style="thin")
    return Border(left=s, right=s, top=s, bottom=s)

def _write(ws, df, status_col=None):
    B = _bdr()
    for c, name in enumerate(df.columns, 1):
        cell = ws.cell(row=1, column=c, value=name)
        cell.fill = HDR; cell.font = WFNT
        cell.alignment = CTR; cell.border = B

    si = list(df.columns).index(status_col)+1 if status_col in (df.columns if status_col else []) else None

    for r, row in enumerate(df.itertuples(index=False), 2):
        sv   = ws.cell(row=r, column=si).value if si else None
        fill = GRN if "✅" in str(sv) else (YLW if "⚠️" in str(sv) else
               (RED if "MISSING" in str(sv) else (ALT if r%2==0 else None)))
        for c, val in enumerate(row, 1):
            cell = ws.cell(row=r, column=c, value=val)
            cell.font = BFNT; cell.alignment = LFT; cell.border = B
            if fill: cell.fill = fill

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    for col in ws.columns:
        w = max((len(str(c.value or "")) for c in col), default=8)
        ws.column_dimensions[get_column_letter(col[0].column)].width = min(w+2, 45)


def export(inventory_df, coverage_df, recommended, out_dir):
    os.makedirs(out_dir, exist_ok=True)

    inventory_df.to_csv(os.path.join(out_dir, "inventory.csv"), index=False)
    coverage_df.to_csv(os.path.join(out_dir, "coverage_summary.csv"), index=False)

    wb  = Workbook()
    ws1 = wb.active; ws1.title = "Source Inventory"
    _write(ws1, inventory_df)

    ws2 = wb.create_sheet("Class Coverage")
    _write(ws2, coverage_df, status_col="status")

    ws3 = wb.create_sheet("Dataset Size Options")
    _write(ws3, pd.DataFrame([
        {"option": "Conservative", "total_clips": recommended["conservative"],
         "cooking": recommended["conservative"]//2,
         "non_cooking": recommended["conservative"]//2,
         "note": "Safe — good starting point"},
        {"option": "Standard ✅", "total_clips": recommended["standard"],
         "cooking": recommended["standard"]//2,
         "non_cooking": recommended["standard"]//2,
         "note": "Recommended — balanced size vs training time"},
        {"option": "Full", "total_clips": recommended["full"],
         "cooking": recommended["full"]//2,
         "non_cooking": recommended["full"]//2,
         "note": "Maximum — best for final submission"},
    ]))

    for name in ["Generation Plan", "Mixed Clips Metadata",
                 "Train Split", "Val Split", "Test Split",
                 "Training Log", "Prediction Results", "Error Analysis"]:
        ws = wb.create_sheet(name)
        ws["A1"] = "← Will be populated in a later phase"
        ws["A1"].font = Font(name="Arial", italic=True, color="888888", size=9)

    xlsx = os.path.join(out_dir, "pipeline_report.xlsx")
    wb.save(xlsx)

    print(f"  ✅ inventory.csv        → {out_dir}")
    print(f"  ✅ coverage_summary.csv → {out_dir}")
    print(f"  ✅ pipeline_report.xlsx → {xlsx}")


# ============================================================
# MAIN
# ============================================================

def main():
    print("\n" + "="*60)
    print("  PHASE 0 — Drive Inventory (v2 fixed paths)")
    print("  Samsung PRISM — Kitchen Audio Dataset v2")
    print(f"  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*60)

    ok = check_structure(RAW_SOURCES_ROOT)
    if not ok:
        print("\n⛔ Fix Drive structure and re-run.")
        return

    inventory_df = build_inventory(RAW_SOURCES_ROOT)
    if inventory_df.empty:
        print("\n⛔ No files found.")
        return

    coverage_df  = class_coverage(inventory_df)
    recommended  = recommend_size(coverage_df)

    print("\n" + "="*60)
    print("  PART E — Saving Output Files")
    print("="*60)
    export(inventory_df, coverage_df, recommended, OUTPUT_DIR)

    print("\n" + "="*60)
    print("  PHASE 0 COMPLETE ✅")
    print("="*60)
    usable = inventory_df[inventory_df["is_usable"]==1].shape[0]
    print(f"\n  Total files  : {len(inventory_df)}")
    print(f"  Usable files : {usable}")
    print(f"\n  Recommended dataset size : {recommended['standard']} clips")
    print(f"  Output → {OUTPUT_DIR}")
    print(f"\n  NEXT → Phase 1: Generation Plan")
    print("="*60)


main()

In [ ]:
# ============================================================
# DRIVE FINDER — Locate existing audio files anywhere in Drive
# Run this in Google Colab BEFORE Phase 0
#
# Paste into a Colab cell and run.
# It will find ALL audio files in your Drive and tell you
# exactly where they are so you can update the path config.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path
from collections import defaultdict

AUDIO_EXTENSIONS = {".wav", ".mp3", ".flac", ".ogg", ".m4a"}

# Where to start scanning — change if your files are in a specific subfolder
SCAN_ROOT = "/content/drive/MyDrive"

# Stop scanning inside these folders (too large, irrelevant)
SKIP_FOLDERS = {"node_modules", ".git", "__pycache__", ".ipynb_checkpoints"}

print("Scanning Google Drive for audio files...")
print(f"Root: {SCAN_ROOT}")
print("(This may take 1-2 minutes depending on Drive size)\n")

# ── Scan ─────────────────────────────────────────────────────
found_files   = []   # (filepath, folder_path)
folder_counts = defaultdict(int)

for root, dirs, files in os.walk(SCAN_ROOT):
    # Skip irrelevant folders
    dirs[:] = [d for d in dirs if d not in SKIP_FOLDERS]

    audio_files = [f for f in files
                   if Path(f).suffix.lower() in AUDIO_EXTENSIONS]

    if audio_files:
        rel_root = root.replace("/content/drive/MyDrive/", "MyDrive/")
        folder_counts[root] = len(audio_files)
        for f in audio_files:
            found_files.append(os.path.join(root, f))

# ── Report ────────────────────────────────────────────────────
print("=" * 65)
print(f"  RESULTS — Audio files found: {len(found_files)}")
print("=" * 65)

if not found_files:
    print("\n❌ No audio files found anywhere in Drive.")
    print("   Your source files may not be uploaded yet.")
    print("\n   You need to upload your raw audio files first:")
    print("   Freesound downloads → MyDrive/kitpri/raw_sources/freesound/frying/")
    print("   FSD50K files        → MyDrive/kitpri/raw_sources/fsd50k/boiling/")
    print("   MUSAN files         → MyDrive/kitpri/raw_sources/musan/speech/")
    print("   ESC-50 files        → MyDrive/kitpri/raw_sources/esc50/street_noise/")

else:
    # Show folders with audio files, sorted by count
    print(f"\n  Folders containing audio files ({len(folder_counts)} folders):\n")
    sorted_folders = sorted(folder_counts.items(), key=lambda x: -x[1])

    for folder, count in sorted_folders:
        rel = folder.replace("/content/drive/MyDrive", "MyDrive")
        # Show sample filenames
        sample_files = [f for f in os.listdir(folder)
                        if Path(f).suffix.lower() in AUDIO_EXTENSIONS][:3]
        samples = ", ".join(sample_files)
        print(f"  📁 {rel}")
        print(f"     {count} files  |  e.g. {samples}")
        print()

    # ── Detect likely structure ───────────────────────────────
    print("=" * 65)
    print("  STRUCTURE DETECTION")
    print("=" * 65)

    # Find the common root of all audio folders
    all_folders  = list(folder_counts.keys())
    common_parts = Path(all_folders[0]).parts
    for folder in all_folders[1:]:
        folder_parts = Path(folder).parts
        common_parts = tuple(
            p for p, q in zip(common_parts, folder_parts) if p == q
        )
    common_root = str(Path(*common_parts)) if common_parts else SCAN_ROOT

    print(f"\n  Likely root of your dataset: {common_root}")
    print(f"\n  Update RAW_SOURCES_ROOT in phase0_drive_inventory.py to:")
    print(f"\n      RAW_SOURCES_ROOT = \"{common_root}\"")

    # ── Check if structure matches expected ───────────────────
    print(f"\n" + "=" * 65)
    print("  FOLDER STRUCTURE ANALYSIS")
    print("=" * 65)

    # Check depth of structure
    depths = defaultdict(list)
    for folder in all_folders:
        rel_parts = Path(folder).relative_to(common_root).parts
        depth     = len(rel_parts)
        depths[depth].append(folder)

    print(f"\n  Folder depth from common root:")
    for depth, folders in sorted(depths.items()):
        print(f"\n  Depth {depth}:")
        for f in folders[:5]:  # show max 5 per depth
            rel = str(Path(f).relative_to(common_root))
            print(f"    {rel}/  ({folder_counts[f]} files)")
        if len(folders) > 5:
            print(f"    ... and {len(folders)-5} more")

    # ── Diagnosis ─────────────────────────────────────────────
    print(f"\n" + "=" * 65)
    print("  DIAGNOSIS")
    print("=" * 65)

    depth_2_folders = depths.get(2, [])
    depth_1_folders = depths.get(1, [])
    depth_0_folders = depths.get(0, [])

    if depth_2_folders:
        # Good — likely source/class/files structure
        print(f"\n  ✅ Your structure looks like Option B (source/class/files)")
        print(f"     This is what Phase 0 expects.")
        print(f"\n     Just update RAW_SOURCES_ROOT to:")
        print(f"         \"{common_root}\"")

    elif depth_1_folders:
        # One level deep — class/files or source/files
        sample = str(Path(depth_1_folders[0]).relative_to(common_root))
        print(f"\n  ⚠️  Your structure is one level deep (e.g. {sample}/files)")
        print(f"     This could be class/files OR source/files.")
        print(f"\n     If folders are CLASS names (frying, chopping, speech...):")
        print(f"       → Move them inside a source folder first:")
        print(f"           {common_root}/freesound/frying/")
        print(f"           {common_root}/freesound/chopping/")
        print(f"\n     If folders are SOURCE names (freesound, fsd50k...):")
        print(f"       → Files are flat inside source folders, need class subfolders")

    elif depth_0_folders:
        # All flat in root
        print(f"\n  ❌ All files are flat in one folder — no subfolders.")
        print(f"     You need to organize them by source/class.")
        print(f"     Example:")
        print(f"       {common_root}/freesound/frying/frying_001.wav")
        print(f"       {common_root}/musan/speech/speech_001.wav")

    else:
        print(f"\n  ℹ️  Could not auto-detect structure.")
        print(f"     Review the folder list above and organize accordingly.")

    print(f"\n" + "=" * 65)
    print(f"  Share the output above and we will update the path config.")
    print("=" * 65)

In [ ]:
# ============================================================
# DRIVE REORGANIZER — Kitchen Audio Dataset v2
# Samsung PRISM
#
# HOW TO RUN:
#   1. Open Google Colab
#   2. Paste this entire script into one cell
#   3. Run it
#
# WHAT IT DOES:
#   Step 1 — Moves musan_speech, tv_speech, music from
#             background/ → foreground/ (they are FG sources)
#   Step 2 — Sorts esc50_misc flat files into class subfolders
#             using ESC-50 category labels from filenames
#   Step 3 — Creates the raw_sources/ structure Phase 0 expects
#   Step 4 — Prints a full report of what moved where
#   Step 5 — DRY RUN by default — set DRY_RUN = False to actually move
#
# SAFETY:
#   Files are COPIED not moved until you set DRY_RUN = False
#   Original files are never deleted unless you confirm
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
from pathlib import Path
from collections import defaultdict

# ============================================================
# CONFIGURATION
# ============================================================

SOUNDBANK_ROOT = "/content/drive/MyDrive/kitpri/soundbank"
RAW_SOURCES_OUT = "/content/drive/MyDrive/kitpri/raw_sources"

# Set to False when you are ready to actually move files
DRY_RUN = False

# ── ESC-50 category mapping ──────────────────────────────────
# ESC-50 filenames are: {fold}-{clip_id}-{take}-{category_id}.wav
# Category IDs 0-49 map to these classes:
# We only keep categories relevant to our background classes.

ESC50_CATEGORY_MAP = {
    # category_id : our_class_name
    # Animals
    0:  "dog_barking",       # dog
    1:  "noncooking_other",  # rooster
    2:  "noncooking_other",  # pig
    3:  "noncooking_other",  # cow
    4:  "noncooking_other",  # frog
    5:  "noncooking_other",  # cat
    6:  "noncooking_other",  # hen
    7:  "noncooking_other",  # insects
    8:  "noncooking_other",  # sheep
    9:  "noncooking_other",  # crow
    # Natural soundscapes
    10: "rain",              # rain
    11: "noncooking_other",  # sea waves
    12: "noncooking_other",  # crackling fire
    13: "noncooking_other",  # crickets
    14: "noncooking_other",  # chirping birds
    15: "noncooking_other",  # water drops
    16: "noncooking_other",  # wind
    17: "noncooking_other",  # pouring water
    18: "noncooking_other",  # toilet flush
    19: "noncooking_other",  # thunderstorm
    # Human non-speech
    20: "noncooking_other",  # crying baby  → could be FG non-cooking
    21: "noncooking_other",  # sneezing
    22: "noncooking_other",  # clapping
    23: "noncooking_other",  # breathing
    24: "noncooking_other",  # coughing
    25: "footsteps",         # footsteps
    26: "noncooking_other",  # laughing
    27: "noncooking_other",  # brushing teeth
    28: "noncooking_other",  # snoring
    29: "noncooking_other",  # drinking sipping
    # Interior/domestic
    30: "door_knock",        # door knock
    31: "noncooking_other",  # mouse click
    32: "keyboard_typing",   # keyboard typing
    33: "noncooking_other",  # door wood creaks
    34: "noncooking_other",  # can opening
    35: "noncooking_other",  # washing machine
    36: "vacuum_cleaner",    # vacuum cleaner
    37: "noncooking_other",  # clock alarm
    38: "noncooking_other",  # clock tick
    39: "phone_ringing",     # glass breaking → repurpose as phone? No — skip
    # Exterior/urban
    40: "noncooking_other",  # helicopter
    41: "noncooking_other",  # chainsaw
    42: "noncooking_other",  # siren
    43: "street_noise",      # car horn
    44: "street_noise",      # engine
    45: "noncooking_other",  # train
    46: "street_noise",      # church bells
    47: "noncooking_other",  # airplane
    48: "street_noise",      # fireworks
    49: "street_noise",      # hand saw → background street noise
}

# Which ESC-50 mapped classes go to foreground vs background
ESC50_FG_CLASSES = {
    "dog_barking", "footsteps", "door_knock",
    "keyboard_typing", "vacuum_cleaner",
}
ESC50_BG_CLASSES = {
    "rain", "street_noise", "noncooking_other",
}

# ── Folder moves: background → foreground ───────────────────
# These folders exist in soundbank/background/ but should be FG sources
BG_TO_FG_MOVES = {
    "musan_speech": "speech",      # 100 files → foreground/speech
    "tv_speech":    "TV_audio",    # 58 files  → foreground/TV_audio
    "music":        "music",       # 40 files  → foreground/music
}

# ── Final raw_sources structure we're building ───────────────
# raw_sources/
#   foreground/
#     cooking/      ← all cooking FG classes
#     noncooking/   ← all non-cooking FG classes
#   background/     ← all background classes

COOKING_FG_CLASSES = [
    "frying", "boiling", "water_tap", "chopping", "stove",
    "dishes", "microwave", "stirring", "blender", "pressure_cooker",
]

NONCOOKING_FG_CLASSES = [
    "speech", "TV_audio", "music", "dog_barking", "keyboard_typing",
    "door_knock", "vacuum_cleaner", "footsteps", "phone_ringing",
]

BACKGROUND_CLASSES = [
    "room_ambience", "fan_ac", "musan_noise",
    "rain", "street_noise",
]


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def safe_copy(src: str, dst: str, dry_run: bool) -> bool:
    """Copy file from src to dst. Creates dst dir if needed."""
    try:
        if not dry_run:
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(src, dst)
        return True
    except Exception as e:
        print(f"      ❌ Failed to copy {Path(src).name}: {e}")
        return False


def count_files(folder: str) -> int:
    if not os.path.exists(folder):
        return 0
    return len([f for f in os.listdir(folder)
                if Path(f).suffix.lower() in {".wav", ".mp3", ".flac", ".ogg"}])


# ============================================================
# STEP 1 — Move bg→fg folders
# ============================================================

def step1_move_bg_to_fg(dry_run: bool) -> dict:
    print("\n" + "="*60)
    print("  STEP 1 — Reclassify background folders → foreground")
    print("="*60)

    results = {}

    for src_folder_name, dst_class_name in BG_TO_FG_MOVES.items():
        src_dir = os.path.join(SOUNDBANK_ROOT, "background", src_folder_name)
        dst_dir = os.path.join(RAW_SOURCES_OUT, "foreground", "noncooking", dst_class_name)

        if not os.path.exists(src_dir):
            print(f"\n  ⚠️  {src_folder_name}/ not found — skipping")
            results[src_folder_name] = 0
            continue

        files = [f for f in os.listdir(src_dir)
                 if Path(f).suffix.lower() in {".wav", ".mp3", ".flac", ".ogg"}]

        print(f"\n  📁 soundbank/background/{src_folder_name}/ → foreground/noncooking/{dst_class_name}/")
        print(f"     {len(files)} files to copy")

        copied = 0
        for fname in files:
            src = os.path.join(src_dir, fname)
            dst = os.path.join(dst_dir, fname)
            if safe_copy(src, dst, dry_run):
                copied += 1

        status = "✅ DRY RUN" if dry_run else "✅ Copied"
        print(f"     {status}: {copied}/{len(files)} files")
        results[src_folder_name] = copied

    return results


# ============================================================
# STEP 2 — Sort ESC-50 flat files into class subfolders
# ============================================================

def parse_esc50_category(filename: str) -> int:
    """
    ESC-50 filename format: {fold}-{clip_id}-{take}-{category}.wav
    e.g. 1-100032-A-0.wav → category 0 → dog
    Returns category int or -1 if unparseable.
    """
    stem = Path(filename).stem   # remove .wav
    parts = stem.split("-")
    if len(parts) >= 4:
        try:
            return int(parts[-1])   # last segment is category
        except ValueError:
            pass
    return -1


def step2_sort_esc50(dry_run: bool) -> dict:
    print("\n" + "="*60)
    print("  STEP 2 — Sort ESC-50 files into class subfolders")
    print("="*60)

    esc50_dir = os.path.join(SOUNDBANK_ROOT, "background", "esc50_misc")

    if not os.path.exists(esc50_dir):
        print(f"\n  ⚠️  esc50_misc/ not found — skipping")
        return {}

    files = [f for f in os.listdir(esc50_dir)
             if Path(f).suffix.lower() in {".wav", ".mp3", ".flac"}]

    print(f"\n  Found {len(files)} files in esc50_misc/")

    results     = defaultdict(int)
    unrecognized = []

    for fname in files:
        cat_id = parse_esc50_category(fname)

        if cat_id == -1 or cat_id not in ESC50_CATEGORY_MAP:
            unrecognized.append(fname)
            continue

        our_class = ESC50_CATEGORY_MAP[cat_id]

        # Decide: FG or BG destination?
        if our_class in ESC50_FG_CLASSES:
            dst_dir = os.path.join(
                RAW_SOURCES_OUT, "foreground", "noncooking", our_class
            )
        else:
            # background or noncooking_other → background/esc50_sorted/
            dst_dir = os.path.join(
                RAW_SOURCES_OUT, "background", our_class
            )

        src = os.path.join(esc50_dir, fname)
        dst = os.path.join(dst_dir, fname)
        if safe_copy(src, dst, dry_run):
            results[our_class] += 1

    # Print results
    print(f"\n  Sorted into classes:")
    for cls, count in sorted(results.items(), key=lambda x: -x[1]):
        dest = "foreground/noncooking" if cls in ESC50_FG_CLASSES else "background"
        print(f"    {cls:<25} {count:>5} files  → {dest}/{cls}/")

    if unrecognized:
        print(f"\n  ⚠️  {len(unrecognized)} files could not be parsed (non-ESC50 format)")
        for f in unrecognized[:5]:
            print(f"      {f}")
        if len(unrecognized) > 5:
            print(f"      ... and {len(unrecognized)-5} more")

    return dict(results)


# ============================================================
# STEP 3 — Copy cooking FG folders into raw_sources structure
# ============================================================

def step3_copy_cooking_fg(dry_run: bool) -> dict:
    print("\n" + "="*60)
    print("  STEP 3 — Copy cooking FG into raw_sources/foreground/cooking/")
    print("="*60)

    results = {}

    for cls in COOKING_FG_CLASSES:
        src_dir = os.path.join(SOUNDBANK_ROOT, "foreground", cls)
        dst_dir = os.path.join(RAW_SOURCES_OUT, "foreground", "cooking", cls)

        if not os.path.exists(src_dir):
            print(f"\n  ⚠️  foreground/{cls}/ not found — skipping")
            results[cls] = 0
            continue

        files = [f for f in os.listdir(src_dir)
                 if Path(f).suffix.lower() in {".wav", ".mp3", ".flac", ".ogg"}]

        copied = 0
        for fname in files:
            src = os.path.join(src_dir, fname)
            dst = os.path.join(dst_dir, fname)
            if safe_copy(src, dst, dry_run):
                copied += 1

        status = "DRY RUN ✅" if dry_run else "Copied ✅"
        print(f"  {cls:<22} {copied:>4} files  → foreground/cooking/{cls}/  {status}")
        results[cls] = copied

    return results


# ============================================================
# STEP 4 — Copy background folders into raw_sources structure
# ============================================================

def step4_copy_backgrounds(dry_run: bool) -> dict:
    print("\n" + "="*60)
    print("  STEP 4 — Copy background folders into raw_sources/background/")
    print("="*60)

    results = {}

    for cls in BACKGROUND_CLASSES:
        src_dir = os.path.join(SOUNDBANK_ROOT, "background", cls)
        dst_dir = os.path.join(RAW_SOURCES_OUT, "background", cls)

        if not os.path.exists(src_dir):
            print(f"  ⚠️  background/{cls}/ not found — skipping")
            results[cls] = 0
            continue

        files = [f for f in os.listdir(src_dir)
                 if Path(f).suffix.lower() in {".wav", ".mp3", ".flac", ".ogg"}]

        copied = 0
        for fname in files:
            src = os.path.join(src_dir, fname)
            dst = os.path.join(dst_dir, fname)
            if safe_copy(src, dst, dry_run):
                copied += 1

        status = "DRY RUN ✅" if dry_run else "Copied ✅"
        print(f"  {cls:<22} {copied:>4} files  → background/{cls}/  {status}")
        results[cls] = copied

    return results


# ============================================================
# STEP 5 — Final report + what's still missing
# ============================================================

def step5_final_report(
    step1_results: dict,
    step2_results: dict,
    step3_results: dict,
    step4_results: dict,
    dry_run: bool,
):
    print("\n" + "="*60)
    print("  STEP 5 — Final Report")
    print("="*60)

    mode = "DRY RUN (nothing actually moved)" if dry_run else "FILES COPIED"
    print(f"\n  Mode: {mode}")

    print(f"\n  ┌─────────────────────────────────────────────────────┐")
    print(f"  │  raw_sources/ structure after reorganization        │")
    print(f"  ├─────────────────────────────────────────────────────┤")

    # Cooking FG
    print(f"  │  foreground/cooking/                                │")
    total_cooking = 0
    for cls in COOKING_FG_CLASSES:
        n = step3_results.get(cls, 0)
        total_cooking += n
        flag = "✅" if n >= 45 else ("⚠️ " if n > 0 else "❌")
        print(f"  │    {cls:<22} {n:>4} files  {flag}               │")

    # Non-cooking FG
    print(f"  ├─────────────────────────────────────────────────────┤")
    print(f"  │  foreground/noncooking/                             │")
    total_noncooking = 0

    # From BG→FG moves
    bg_to_fg_map = {"musan_speech": "speech", "tv_speech": "TV_audio", "music": "music"}
    moved_classes = {}
    for src, dst in bg_to_fg_map.items():
        moved_classes[dst] = step1_results.get(src, 0)

    # From ESC-50
    esc_fg_classes = {c: step2_results.get(c, 0) for c in ESC50_FG_CLASSES}

    all_noncooking = {**moved_classes, **esc_fg_classes}
    for cls in NONCOOKING_FG_CLASSES:
        n = all_noncooking.get(cls, 0)
        total_noncooking += n
        flag = "✅" if n >= 40 else ("⚠️ " if n > 0 else "❌ NEED TO DOWNLOAD")
        print(f"  │    {cls:<22} {n:>4} files  {flag}               │")

    # Background
    print(f"  ├─────────────────────────────────────────────────────┤")
    print(f"  │  background/                                        │")
    total_bg = 0
    all_bg = {**step4_results}
    for cls, n in step2_results.items():
        if cls not in ESC50_FG_CLASSES:
            all_bg[cls] = all_bg.get(cls, 0) + n

    for cls, n in sorted(all_bg.items(), key=lambda x: -x[1]):
        total_bg += n
        flag = "✅" if n >= 40 else ("⚠️ " if n > 0 else "❌")
        print(f"  │    {cls:<22} {n:>4} files  {flag}               │")

    print(f"  ├─────────────────────────────────────────────────────┤")
    print(f"  │  TOTAL COOKING FG    : {total_cooking:>5} files                   │")
    print(f"  │  TOTAL NON-COOK FG   : {total_noncooking:>5} files                   │")
    print(f"  │  TOTAL BACKGROUND    : {total_bg:>5} files                   │")
    print(f"  └─────────────────────────────────────────────────────┘")

    # What still needs downloading
    print(f"\n  Classes that still need files downloaded:")
    needs_download = False
    for cls in NONCOOKING_FG_CLASSES:
        n = all_noncooking.get(cls, 0)
        if n < 40:
            needed = 40 - n
            print(f"  ❌ {cls:<22} has {n:>3} files — download {needed} more")
            print(f"     → Search Freesound.org: \"{cls.replace('_', ' ')} sound\"")
            needs_download = True

    for cls in COOKING_FG_CLASSES:
        n = step3_results.get(cls, 0)
        if n < 20:
            needed = 40 - n
            print(f"  ⚠️  {cls:<22} has {n:>3} files — download {needed} more")
            needs_download = True

    if not needs_download:
        print(f"  ✅ All classes have sufficient files — ready for Phase 0!")

    if dry_run:
        print(f"\n" + "="*60)
        print(f"  ⚠️  DRY RUN COMPLETE — nothing was actually copied")
        print(f"  Review the report above.")
        print(f"  If everything looks correct, set DRY_RUN = False")
        print(f"  and run again to actually copy the files.")
        print("="*60)
    else:
        print(f"\n" + "="*60)
        print(f"  ✅ REORGANIZATION COMPLETE")
        print(f"  raw_sources/ is ready.")
        print(f"  Next step → Run phase0_drive_inventory.py")
        print(f"  with RAW_SOURCES_ROOT = \"{RAW_SOURCES_OUT}\"")
        print("="*60)


# ============================================================
# MAIN
# ============================================================

def main():
    print("="*60)
    print("  DRIVE REORGANIZER — Samsung PRISM Kitchen Audio v2")
    print("="*60)
    print(f"\n  Source  : {SOUNDBANK_ROOT}")
    print(f"  Output  : {RAW_SOURCES_OUT}")
    print(f"  Mode    : {'⚠️  DRY RUN — no files will be moved' if DRY_RUN else '🚀 LIVE — files will be copied'}")

    if not os.path.exists(SOUNDBANK_ROOT):
        print(f"\n❌ SOUNDBANK_ROOT not found: {SOUNDBANK_ROOT}")
        print("   Update the path at the top of this script.")
        return

    # Create output root
    if not DRY_RUN:
        os.makedirs(RAW_SOURCES_OUT, exist_ok=True)

    r1 = step1_move_bg_to_fg(DRY_RUN)
    r2 = step2_sort_esc50(DRY_RUN)
    r3 = step3_copy_cooking_fg(DRY_RUN)
    r4 = step4_copy_backgrounds(DRY_RUN)
    step5_final_report(r1, r2, r3, r4, DRY_RUN)


main()

In [ ]:
from getpass import getpass
API_KEY = getpass("Enter your Freesound API key: ")

In [ ]:
# ============================================================
# FREESOUND DOWNLOADER v3 — API Key Only (No OAuth)
# Samsung PRISM — Kitchen Audio Dataset v2
#
# Uses API key for search + HQ preview MP3 downloads.
# No OAuth flow needed. Previews are 128kbps MP3 —
# librosa handles them identically to WAV during mixing.
#
# HOW TO RUN:
#   1. Paste into a Colab cell and run
#   2. Enter your API Key when prompted (long string from freesound.org/apiv2/apply/)
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import subprocess
subprocess.run(["pip", "install", "requests", "-q"], check=True)

import os
import time
import requests
from pathlib import Path
from getpass import getpass

# ============================================================
# CREDENTIALS
# ============================================================

print("="*55)
print("  Enter your Freesound API Key")
print("  (the long string — 'Client secret/Api key' column)")
print("  from freesound.org/apiv2/apply/")
print("="*55)
API_KEY = getpass("API Key: ")

# ============================================================
# CONFIGURATION
# ============================================================

SOUNDBANK_ROOT = "/content/drive/MyDrive/kitpri/soundbank/foreground"
BASE_URL       = "https://freesound.org/apiv2"
MIN_DURATION   = 1.0
MAX_DURATION   = 30.0
DOWNLOAD_PAUSE = 0.3   # seconds between requests

DOWNLOAD_TARGETS = [
    {
        "class_name":   "vacuum_cleaner",
        "folder":       "vacuum_cleaner",
        "queries":      ["vacuum cleaner", "hoover vacuum", "vacuum cleaning"],
        "target":       40,
        "already_have": 0,
    },
    {
        "class_name":   "phone_ringing",
        "folder":       "phone_ringing",
        "queries":      ["phone ringing", "telephone ring", "mobile phone ring"],
        "target":       40,
        "already_have": 0,
    },
    {
        "class_name":   "pressure_cooker",
        "folder":       "pressure_cooker",
        "queries":      ["pressure cooker", "pressure cooker steam", "cooker whistle"],
        "target":       40,
        "already_have": 20,
    },
]


# ============================================================
# API FUNCTIONS
# ============================================================

def validate_key(api_key: str) -> bool:
    """Quick search test to confirm key works."""
    print("\nValidating API key...")
    url    = f"{BASE_URL}/search/text/"
    params = {"query": "test", "token": api_key, "page_size": 1,
              "fields": "id,name"}
    try:
        r = requests.get(url, params=params, timeout=15)
        if r.status_code == 200:
            print("✅ API key valid")
            return True
        elif r.status_code == 401:
            print("❌ Invalid API key — copy the long 'Client secret/Api key' value")
            return False
        else:
            print(f"❌ Unexpected status {r.status_code}: {r.text[:100]}")
            return False
    except Exception as e:
        print(f"❌ Connection error: {e}")
        return False


def search_sounds(query: str, api_key: str, page_size: int = 150) -> list:
    """Search Freesound and return list of sound dicts."""
    url    = f"{BASE_URL}/search/text/"
    params = {
        "query":      query,
        "token":      api_key,
        "page_size":  page_size,
        "fields":     "id,name,duration,previews,type,license",
        "filter":     f"duration:[{MIN_DURATION} TO {MAX_DURATION}]",
        "sort":       "score",
    }
    try:
        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()
        results = r.json().get("results", [])
        return results
    except requests.exceptions.HTTPError:
        if r.status_code == 429:
            print("    ⚠️  Rate limited — waiting 60s...")
            time.sleep(60)
            return search_sounds(query, api_key, page_size)
        print(f"    ❌ Search failed {r.status_code}: {r.text[:80]}")
        return []
    except Exception as e:
        print(f"    ❌ Search error: {e}")
        return []


def download_preview_mp3(sound: dict, save_path: str) -> bool:
    """
    Download HQ preview MP3 (128kbps).
    No authentication required — works with any API key.
    Saved as .mp3 — librosa loads these identically to WAV.
    """
    preview_url = sound.get("previews", {}).get("preview-hq-mp3", "")
    if not preview_url:
        # fallback to LQ preview
        preview_url = sound.get("previews", {}).get("preview-lq-mp3", "")
    if not preview_url:
        return False

    save_path = str(Path(save_path).with_suffix(".mp3"))

    try:
        r = requests.get(preview_url, stream=True, timeout=30)
        if r.status_code == 200:
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            with open(save_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
            return True
        return False
    except Exception:
        return False


# ============================================================
# DOWNLOAD LOOP
# ============================================================

def download_class(target: dict, api_key: str) -> dict:
    class_name = target["class_name"]
    folder     = target["folder"]
    queries    = target["queries"]
    need       = target["target"] - target["already_have"]
    save_dir   = os.path.join(SOUNDBANK_ROOT, folder)

    print(f"\n{'='*55}")
    print(f"  Class  : {class_name}")
    print(f"  Need   : {need} files")
    print(f"  Save   : {save_dir}")
    print(f"{'='*55}")

    if need <= 0:
        print("  ✅ Already sufficient — skipping")
        return {"class": class_name, "downloaded": 0, "skipped": True}

    os.makedirs(save_dir, exist_ok=True)

    # Track existing files to avoid duplicates
    existing_ids = set()
    for f in os.listdir(save_dir):
        if Path(f).suffix.lower() in {".wav", ".mp3", ".flac", ".ogg"}:
            # filename format: classname_soundid.ext
            parts = Path(f).stem.split("_")
            if parts[-1].isdigit():
                existing_ids.add(int(parts[-1]))

    print(f"  Already in folder: {len(existing_ids)} files")

    downloaded = 0
    failed     = 0
    seen_ids   = set(existing_ids)

    # Try each query until we have enough
    for query in queries:
        if downloaded >= need:
            break

        print(f"\n  Searching: \"{query}\"")
        results = search_sounds(query, api_key)
        print(f"  → {len(results)} candidates")

        for sound in results:
            if downloaded >= need:
                break

            sound_id = sound["id"]
            name     = sound["name"]
            duration = sound.get("duration", 0)

            # Skip duplicates and bad durations
            if sound_id in seen_ids:
                continue
            if not (MIN_DURATION <= duration <= MAX_DURATION):
                continue

            seen_ids.add(sound_id)
            save_path = os.path.join(save_dir, f"{class_name}_{sound_id}.mp3")

            print(f"  [{downloaded+1:>2}/{need}] {name[:48]:<48} ({duration:.1f}s) ", end="")

            ok = download_preview_mp3(sound, save_path)

            if ok:
                downloaded += 1
                print("✅")
            else:
                failed += 1
                print("❌")

            time.sleep(DOWNLOAD_PAUSE)

    total_now = len(existing_ids) + downloaded
    print(f"\n  ── Result ───────────────────────────────────")
    print(f"  Downloaded : {downloaded}/{need}")
    print(f"  Failed     : {failed}")
    print(f"  Total now  : {total_now} files in {folder}/")

    if downloaded < need:
        shortfall = need - downloaded
        print(f"  ⚠️  Still short by {shortfall} — try manual download:")
        print(f"     https://freesound.org/search/?q={queries[0].replace(' ', '+')}")

    return {"class": class_name, "downloaded": downloaded,
            "failed": failed, "total": total_now}


# ============================================================
# MAIN
# ============================================================

def main():
    print("="*55)
    print("  FREESOUND DOWNLOADER v3 — Samsung PRISM")
    print("="*55)

    if not validate_key(API_KEY):
        return

    results = []
    for target in DOWNLOAD_TARGETS:
        r = download_class(target, API_KEY)
        results.append(r)

    # Final summary
    print(f"\n{'='*55}")
    print(f"  FINAL SUMMARY")
    print(f"{'='*55}")
    total_new = 0
    for r in results:
        if r.get("skipped"):
            print(f"  {r['class']:<22} — sufficient ✅")
        else:
            flag = "✅" if r["failed"] == 0 else "⚠️"
            print(f"  {r['class']:<22} — {r['downloaded']} new files  "
                  f"(total: {r.get('total','-')})  {flag}")
            total_new += r["downloaded"]

    print(f"\n  Total new files : {total_new}")
    print(f"\n  NEXT STEPS:")
    print(f"  1. Set DRY_RUN = False in drive_reorganizer.py → run")
    print(f"  2. Run phase0_drive_inventory.py → confirm counts")
    print("="*55)


main()

In [ ]:
import shutil, os

SOUNDBANK = "/content/drive/MyDrive/kitpri/soundbank/foreground"
RAW       = "/content/drive/MyDrive/kitpri/raw_sources/foreground/noncooking"

# Copy vacuum_cleaner and phone_ringing into raw_sources
for cls in ["vacuum_cleaner", "phone_ringing"]:
    src = os.path.join(SOUNDBANK, cls)
    dst = os.path.join(RAW, cls)
    os.makedirs(dst, exist_ok=True)
    copied = 0
    for f in os.listdir(src):
        if f.endswith((".wav", ".mp3", ".flac", ".ogg")):
            shutil.copy2(os.path.join(src, f), os.path.join(dst, f))
            copied += 1
    print(f"✅ {cls}: {copied} files → {dst}")

In [ ]:
# ============================================================
# PRESSURE COOKER TOP-UP DOWNLOADER
# Samsung PRISM — Kitchen Audio Dataset v2
#
# Downloads more pressure cooker sounds from Freesound
# using multiple search queries to maximize variety.
# Target: 80 total files (currently have 23, need 57 more)
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os, time, requests
from pathlib import Path
from getpass import getpass

# ── Config ───────────────────────────────────────────────────
API_KEY    = getpass("Freesound API Key: ")
SAVE_DIR   = "/content/drive/MyDrive/kitpri/soundbank/foreground/pressure_cooker"
RAW_DIR    = "/content/drive/MyDrive/kitpri/raw_sources/foreground/cooking/pressure_cooker"
TARGET     = 80
BASE_URL   = "https://freesound.org/apiv2"
MIN_DUR    = 1.5
MAX_DUR    = 30.0
AUDIO_EXT  = {".wav", ".mp3", ".flac", ".ogg", ".m4a"}

# Multiple queries — ordered from most specific to broadest
# to maximize acoustic variety across the 57 files needed
QUERIES = [
    "pressure cooker",
    "pressure cooker steam",
    "pressure cooker whistle",
    "pressure cooker hissing",
    "pressure valve steam",
    "steam release cooking",
    "cooker steam hiss",
    "cooking steam pressure",
    "steam whistle kitchen",
    "pot pressure steam",
    "steam vent hissing",
    "boiling pressure steam",
]

# ── Helpers ───────────────────────────────────────────────────

def count_existing(folder):
    if not os.path.exists(folder):
        return set(), 0
    ids = set()
    for f in os.listdir(folder):
        if Path(f).suffix.lower() in AUDIO_EXT:
            parts = Path(f).stem.split("_")
            if parts[-1].isdigit():
                ids.add(int(parts[-1]))
    return ids, len(ids)


def search(query, api_key, page_size=150):
    params = {
        "query":      query,
        "token":      api_key,
        "page_size":  page_size,
        "fields":     "id,name,duration,previews,type",
        "filter":     f"duration:[{MIN_DUR} TO {MAX_DUR}]",
        "sort":       "score",
    }
    try:
        r = requests.get(f"{BASE_URL}/search/text/",
                         params=params, timeout=30)
        r.raise_for_status()
        return r.json().get("results", [])
    except Exception as e:
        print(f"    Search error: {e}")
        return []


def download_preview(sound, save_path):
    url = sound.get("previews", {}).get("preview-hq-mp3", "")
    if not url:
        url = sound.get("previews", {}).get("preview-lq-mp3", "")
    if not url:
        return False
    save_path = str(Path(save_path).with_suffix(".mp3"))
    try:
        r = requests.get(url, stream=True, timeout=30)
        if r.status_code == 200:
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            with open(save_path, "wb") as f:
                for chunk in r.iter_content(8192):
                    f.write(chunk)
            return True
    except Exception:
        pass
    return False


# ── Validate key ──────────────────────────────────────────────

def validate(api_key):
    r = requests.get(f"{BASE_URL}/search/text/",
                     params={"query": "test", "token": api_key,
                             "page_size": 1, "fields": "id"},
                     timeout=10)
    if r.status_code == 200:
        print("✅ API key valid\n")
        return True
    print(f"❌ API key invalid (status {r.status_code})")
    return False


# ── Main ──────────────────────────────────────────────────────

def main():
    print("="*55)
    print("  Pressure Cooker Top-Up Downloader")
    print("="*55)

    if not validate(API_KEY):
        return

    # Count what we already have in BOTH locations
    ids_soundbank, n_soundbank = count_existing(SAVE_DIR)
    ids_raw,       n_raw       = count_existing(RAW_DIR)
    existing_ids = ids_soundbank | ids_raw
    already_have = len(existing_ids)

    print(f"  Already have: {already_have} files")
    print(f"    soundbank/: {n_soundbank}")
    print(f"    raw_sources/: {n_raw}")
    print(f"  Target      : {TARGET}")
    need = TARGET - already_have

    if need <= 0:
        print(f"\n✅ Already at {already_have} files — no download needed!")
        return

    print(f"  Need        : {need} more files\n")

    os.makedirs(SAVE_DIR, exist_ok=True)

    downloaded = 0
    failed     = 0
    seen_ids   = set(existing_ids)

    for query in QUERIES:
        if downloaded >= need:
            break

        print(f"  Searching: \"{query}\"")
        results = search(query, API_KEY)
        new_results = [s for s in results
                       if s["id"] not in seen_ids
                       and MIN_DUR <= s.get("duration", 0) <= MAX_DUR]
        print(f"  → {len(new_results)} new candidates")

        for sound in new_results:
            if downloaded >= need:
                break

            sound_id = sound["id"]
            name     = sound["name"]
            duration = sound.get("duration", 0)
            seen_ids.add(sound_id)

            save_path = os.path.join(SAVE_DIR,
                                     f"pressure_cooker_{sound_id}.mp3")

            print(f"  [{already_have + downloaded + 1:>2}/{TARGET}] "
                  f"{name[:50]:<50} ({duration:.1f}s) ", end="")

            ok = download_preview(sound, save_path)
            if ok:
                downloaded += 1
                print("✅")
            else:
                failed += 1
                print("❌")

            time.sleep(0.3)

        if new_results:
            print()

    # Copy new files to raw_sources too
    print(f"\n  Syncing new files to raw_sources/...")
    import shutil
    os.makedirs(RAW_DIR, exist_ok=True)
    synced = 0
    for f in os.listdir(SAVE_DIR):
        if Path(f).suffix.lower() in AUDIO_EXT:
            dst = os.path.join(RAW_DIR, f)
            if not os.path.exists(dst):
                shutil.copy2(os.path.join(SAVE_DIR, f), dst)
                synced += 1
    print(f"  Synced {synced} new files to raw_sources/")

    # Final count
    _, final_count = count_existing(RAW_DIR)
    print(f"\n{'='*55}")
    print(f"  COMPLETE")
    print(f"{'='*55}")
    print(f"  Downloaded : {downloaded}")
    print(f"  Failed     : {failed}")
    print(f"  Total in raw_sources/ : {final_count}")

    if final_count >= TARGET:
        print(f"\n  ✅ Target of {TARGET} reached!")
        print(f"  pressure_cooker is no longer the bottleneck.")
        print(f"\n  NEXT STEP → Re-run phase0_v2.py to confirm counts")
        print(f"  then proceed to Phase 1: Generation Plan")
    else:
        shortfall = TARGET - final_count
        print(f"\n  ⚠️  Still short by {shortfall} files.")
        print(f"  Freesound may not have more pressure cooker sounds.")
        print(f"  Options:")
        print(f"    A) Proceed with {final_count} files "
              f"(weakest class bottleneck reduced)")
        print(f"    B) Download 'steam cooking' or 'pot steam' sounds manually")
        print(f"       https://freesound.org/search/?q=pressure+cooker")
    print("="*55)


main()


#PHASE 1


In [ ]:
"""
phase1_generation_plan.py
Samsung PRISM — Kitchen Audio Dataset v2
Phase 1: Generation Plan

Reads inventory.csv, generates 2,500 clip recipes, saves generation_plan.csv,
and validates the plan. Run on Google Colab CPU.

Author: Samsung PRISM Team
Seed: 42 (fully reproducible)
"""

# ─────────────────────────────────────────────────────────────────────────────
# 0. INSTALL & IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import os
import sys
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from collections import defaultdict
from pathlib import Path

try:
    from openpyxl import load_workbook
    from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
    OPENPYXL_OK = True
except ImportError:
    print("[INFO] openpyxl not found — installing...")
    os.system("pip install openpyxl -q")
    from openpyxl import load_workbook
    from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
    OPENPYXL_OK = True

# ─────────────────────────────────────────────────────────────────────────────
# 1. MOUNT GOOGLE DRIVE
# ─────────────────────────────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
    print("[OK] Google Drive mounted.")
except ModuleNotFoundError:
    DRIVE_ROOT = os.path.expanduser("~/kitpri_dev")  # local dev fallback
    print(f"[INFO] Not in Colab — using local path: {DRIVE_ROOT}")

# ─────────────────────────────────────────────────────────────────────────────
# 2. PATHS
# ─────────────────────────────────────────────────────────────────────────────
META_DIR    = Path(DRIVE_ROOT) / "kitpri" / "kitpri_v2" / "metadata"
INV_CSV     = META_DIR / "inventory.csv"
PLAN_CSV    = META_DIR / "generation_plan.csv"
REPORT_XLS  = META_DIR / "pipeline_report.xlsx"

META_DIR.mkdir(parents=True, exist_ok=True)

assert INV_CSV.exists(), f"[ERROR] inventory.csv not found at {INV_CSV}"
print(f"[OK] inventory.csv found: {INV_CSV}")

# ─────────────────────────────────────────────────────────────────────────────
# 3. CLASS DEFINITIONS (locked)
# ─────────────────────────────────────────────────────────────────────────────
COOKING_FG_CLASSES = [
    "frying", "chopping", "boiling", "blender", "stove",
    "dishes", "microwave", "pressure_cooker", "water_tap", "stirring",
]

NONCOOKING_FG_CLASSES = [
    "speech", "TV_audio", "music", "keyboard_typing", "dog_barking",
    "phone_ringing", "door_knock", "vacuum_cleaner", "footsteps",
]

BACKGROUND_CLASSES = [
    "room_ambience", "fan_ac", "musan_noise",
    "street_noise", "rain", "noncooking_other",
]

ALL_FG_CLASSES   = COOKING_FG_CLASSES + NONCOOKING_FG_CLASSES
ALL_CLASSES      = ALL_FG_CLASSES + BACKGROUND_CLASSES

# ─────────────────────────────────────────────────────────────────────────────
# 4. DATASET DESIGN CONSTANTS (locked)
# ─────────────────────────────────────────────────────────────────────────────
TOTAL_CLIPS        = 2500
COOKING_CLIPS      = 1250
NONCOOKING_CLIPS   = 1250
MAX_REUSE          = 3
SEED               = 42

# num_stems distribution: 80% 3-stem, 20% 4-stem
STEM_PROBS         = {3: 0.80, 4: 0.20}

# SNR distribution (stem1 vs stem3)
SNR_BANDS = [
    {"label": "very_hard", "low": -5,  "high":  0, "prob": 0.10},
    {"label": "hard",      "low":  0,  "high":  5, "prob": 0.20},
    {"label": "medium",    "low":  5,  "high": 10, "prob": 0.30},
    {"label": "easy",      "low": 10,  "high": 20, "prob": 0.30},
    {"label": "very_easy", "low": 20,  "high": 30, "prob": 0.10},
]

# Train/val/test split ratios
SPLIT_RATIOS = {"train": 0.70, "val": 0.15, "test": 0.15}

# ─────────────────────────────────────────────────────────────────────────────
# 5. LOAD INVENTORY
# ─────────────────────────────────────────────────────────────────────────────
print("\n[Phase 1] Loading inventory.csv ...")
inv = pd.read_csv(INV_CSV)

# Normalise column names (strip whitespace)
inv.columns = [c.strip() for c in inv.columns]

# Required columns
REQUIRED_COLS = {"file_path", "audio_class"}
missing = REQUIRED_COLS - set(inv.columns)
assert not missing, f"[ERROR] Missing columns in inventory.csv: {missing}"

print(f"  Total rows in inventory: {len(inv)}")

# Build per-class file pools
file_pool: dict[str, list[str]] = defaultdict(list)
for _, row in inv.iterrows():
    cls  = str(row["audio_class"]).strip()
    path = str(row["file_path"]).strip()
    if cls in ALL_CLASSES:
        file_pool[cls].append(path)

print("\n  File pool sizes per class:")
for cls in ALL_CLASSES:
    n = len(file_pool.get(cls, []))
    flag = "  [EMPTY!]" if n == 0 else ""
    print(f"    {cls:<22}: {n:>4} files  (max clips from this class: {n * MAX_REUSE}){flag}")

# Warn if any class is empty
empty = [c for c in ALL_CLASSES if not file_pool.get(c)]
if empty:
    print(f"\n  [WARNING] The following classes have NO files: {empty}")
    print("  Clips requiring those classes will be skipped. Check inventory.csv.\n")

# ─────────────────────────────────────────────────────────────────────────────
# 6. HELPER: REUSE TRACKER & FILE SAMPLER
# ─────────────────────────────────────────────────────────────────────────────
reuse_count: dict[str, int] = defaultdict(int)  # file_path → times used

def available_files(cls: str) -> list[str]:
    """Return files in class that haven't hit the reuse limit."""
    return [f for f in file_pool.get(cls, []) if reuse_count[f] < MAX_REUSE]

def pick_file(cls: str, rng: random.Random, exclude: list[str] = None) -> str | None:
    """Pick a random available file from cls, excluding given paths."""
    candidates = available_files(cls)
    if exclude:
        candidates = [f for f in candidates if f not in exclude]
    if not candidates:
        return None
    chosen = rng.choice(candidates)
    reuse_count[chosen] += 1
    return chosen

# ─────────────────────────────────────────────────────────────────────────────
# 7. HELPER: SNR SAMPLER
# ─────────────────────────────────────────────────────────────────────────────
def pick_snr(rng: random.Random) -> tuple[float, str]:
    """Sample an SNR value and return (snr_dB, band_label)."""
    r = rng.random()
    cumulative = 0.0
    for band in SNR_BANDS:
        cumulative += band["prob"]
        if r <= cumulative:
            snr = rng.uniform(band["low"], band["high"])
            return round(snr, 2), band["label"]
    # Fallback (floating point edge case)
    snr = rng.uniform(SNR_BANDS[-1]["low"], SNR_BANDS[-1]["high"])
    return round(snr, 2), SNR_BANDS[-1]["label"]

# ─────────────────────────────────────────────────────────────────────────────
# 8. HELPER: GAIN CALCULATOR
# ─────────────────────────────────────────────────────────────────────────────
def compute_gains(snr_db: float, rng: random.Random, num_stems: int) -> dict:
    """
    stem1_gain  = U(-3, 0) dB
    stem2_gain  = stem1_gain − U(3, 9) dB
    stem3_gain  = stem1_gain − SNR_dB   (so SNR = stem1 − stem3)
    stem4_gain  = stem3_gain − U(12, 18) dB
    """
    g1 = round(rng.uniform(-3.0, 0.0), 2)
    g2 = round(g1 - rng.uniform(3.0, 9.0), 2)
    g3 = round(g1 - snr_db, 2)
    g4 = round(g3 - rng.uniform(12.0, 18.0), 2) if num_stems == 4 else None
    return {"stem1": g1, "stem2": g2, "stem3": g3, "stem4": g4}

# ─────────────────────────────────────────────────────────────────────────────
# 9. HELPER: STRATIFIED SPLIT
# ─────────────────────────────────────────────────────────────────────────────
def assign_splits(df: pd.DataFrame, rng: random.Random) -> pd.Series:
    """
    Stratify by (label, snr_band, num_stems).
    70% train / 15% val / 15% test.
    """
    splits = pd.Series(index=df.index, dtype=str)
    for _, grp in df.groupby(["label", "snr_band", "num_stems"]):
        indices = grp.index.tolist()
        rng.shuffle(indices)
        n = len(indices)
        n_val  = max(1, round(n * SPLIT_RATIOS["val"]))
        n_test = max(1, round(n * SPLIT_RATIOS["test"]))
        splits[indices[:n_val]]             = "val"
        splits[indices[n_val:n_val+n_test]] = "test"
        splits[indices[n_val+n_test:]]      = "train"
    return splits

# ─────────────────────────────────────────────────────────────────────────────
# 10. GENERATE CLIP RECIPES
# ─────────────────────────────────────────────────────────────────────────────
print("\n[Phase 1] Generating clip recipes ...")

rng = random.Random(SEED)
np.random.seed(SEED)

records = []
skipped = 0

def make_clips(label: int, fg_classes: list[str], count: int):
    global skipped
    made = 0
    attempts = 0
    max_attempts = count * 10  # safety valve

    while made < count and attempts < max_attempts:
        attempts += 1

        # ── num_stems ─────────────────────────────────────────────────────
        num_stems = 3 if rng.random() < STEM_PROBS[3] else 4

        # ── stem1: primary FG ─────────────────────────────────────────────
        s1_class = rng.choice(fg_classes)
        s1_file  = pick_file(s1_class, rng)
        if s1_file is None:
            skipped += 1
            continue

        # ── stem2: secondary FG (different class, same category) ──────────
        other_fg = [c for c in fg_classes if c != s1_class]
        if not other_fg:
            skipped += 1
            continue
        rng.shuffle(other_fg)
        s2_class, s2_file = None, None
        for candidate_class in other_fg:
            f = pick_file(candidate_class, rng, exclude=[s1_file])
            if f:
                s2_class = candidate_class
                s2_file  = f
                break
        if s2_file is None:
            skipped += 1
            continue

        # ── stem3: primary background ─────────────────────────────────────
        bg_classes_avail = [c for c in BACKGROUND_CLASSES if available_files(c)]
        if not bg_classes_avail:
            skipped += 1
            continue
        s3_class = rng.choice(bg_classes_avail)
        s3_file  = pick_file(s3_class, rng)
        if s3_file is None:
            skipped += 1
            continue

        # ── stem4: secondary background (only for 4-stem, different class) ─
        s4_class, s4_file = None, None
        if num_stems == 4:
            other_bg = [c for c in BACKGROUND_CLASSES if c != s3_class and available_files(c)]
            if other_bg:
                s4_class = rng.choice(other_bg)
                s4_file  = pick_file(s4_class, rng, exclude=[s3_file])
            if s4_file is None:
                # Downgrade to 3-stem rather than skip
                num_stems = 3
                s4_class, s4_file = None, None

        # ── SNR & gains ───────────────────────────────────────────────────
        snr_db, snr_band = pick_snr(rng)
        gains = compute_gains(snr_db, rng, num_stems)

        # ── file_id ───────────────────────────────────────────────────────
        file_id = f"{'c' if label == 1 else 'n'}_{made+1:05d}"

        records.append({
            "file_id":       file_id,
            "label":         label,
            "num_stems":     num_stems,
            "split":         "",          # filled in later
            "SNR_dB":        snr_db,
            "snr_band":      snr_band,    # helper col for stratification
            "stem1_class":   s1_class,
            "stem1_file":    s1_file,
            "stem1_role":    "fg_primary",
            "stem1_gain_dB": gains["stem1"],
            "stem2_class":   s2_class,
            "stem2_file":    s2_file,
            "stem2_role":    "fg_secondary",
            "stem2_gain_dB": gains["stem2"],
            "stem3_class":   s3_class,
            "stem3_file":    s3_file,
            "stem3_role":    "bg_primary",
            "stem3_gain_dB": gains["stem3"],
            "stem4_class":   s4_class   if num_stems == 4 else "",
            "stem4_file":    s4_file    if num_stems == 4 else "",
            "stem4_role":    "bg_secondary" if num_stems == 4 else "",
            "stem4_gain_dB": gains["stem4"] if num_stems == 4 else "",
            "mixing_method": "librosa",
            "status":        "pending",
        })
        made += 1

    if made < count:
        print(f"  [WARNING] Only generated {made}/{count} clips for label={label} "
              f"(hit reuse limit or empty classes). Increase source files if needed.")

print("  Generating cooking clips   (label=1) ...")
make_clips(label=1, fg_classes=COOKING_FG_CLASSES,    count=COOKING_CLIPS)
print("  Generating noncooking clips (label=0) ...")
make_clips(label=0, fg_classes=NONCOOKING_FG_CLASSES, count=NONCOOKING_CLIPS)

df = pd.DataFrame(records)
print(f"\n  Total recipes generated: {len(df)}  (skipped: {skipped})")

# ─────────────────────────────────────────────────────────────────────────────
# 11. ASSIGN TRAIN/VAL/TEST SPLITS
# ─────────────────────────────────────────────────────────────────────────────
print("\n[Phase 1] Assigning stratified splits ...")
split_rng = random.Random(SEED + 1)
df["split"] = assign_splits(df, split_rng)

# ─────────────────────────────────────────────────────────────────────────────
# 12. SAVE generation_plan.csv
# ─────────────────────────────────────────────────────────────────────────────
# Keep only the columns specified in the spec (drop helper col snr_band)
OUTPUT_COLS = [
    "file_id", "label", "num_stems", "split", "SNR_dB",
    "stem1_class", "stem1_file", "stem1_role", "stem1_gain_dB",
    "stem2_class", "stem2_file", "stem2_role", "stem2_gain_dB",
    "stem3_class", "stem3_file", "stem3_role", "stem3_gain_dB",
    "stem4_class", "stem4_file", "stem4_role", "stem4_gain_dB",
    "mixing_method", "status",
]
df_out = df[OUTPUT_COLS]
df_out.to_csv(PLAN_CSV, index=False)
print(f"  Saved: {PLAN_CSV}  ({len(df_out)} rows)")

# ─────────────────────────────────────────────────────────────────────────────
# 13. VALIDATION REPORT
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "═" * 60)
print("  PHASE 1 VALIDATION REPORT")
print("═" * 60)

# ── 13a. Clip counts by label ──────────────────────────────────────────────
print("\n▸ Clip counts by label:")
label_counts = df["label"].value_counts().sort_index()
for lbl, cnt in label_counts.items():
    name = "cooking" if lbl == 1 else "non-cooking"
    target = COOKING_CLIPS if lbl == 1 else NONCOOKING_CLIPS
    status = "✓" if cnt == target else f"✗ (target={target})"
    print(f"    label={lbl} ({name}): {cnt}  {status}")
total_ok = len(df) == TOTAL_CLIPS
print(f"    TOTAL: {len(df)} / {TOTAL_CLIPS}  {'✓' if total_ok else '✗'}")

# ── 13b. Split ratios ─────────────────────────────────────────────────────
print("\n▸ Split distribution:")
split_counts = df["split"].value_counts()
for split in ["train", "val", "test"]:
    cnt = split_counts.get(split, 0)
    frac = cnt / len(df)
    target = SPLIT_RATIOS[split]
    ok = abs(frac - target) < 0.03
    print(f"    {split:<6}: {cnt:>5}  ({frac:.1%})  target={target:.0%}  {'✓' if ok else '~'}")

# ── 13c. Reuse limit check ────────────────────────────────────────────────
print("\n▸ File reuse limit (max 3×):")
violations = {f: c for f, c in reuse_count.items() if c > MAX_REUSE}
if violations:
    print(f"    ✗ {len(violations)} files exceeded limit:")
    for f, c in list(violations.items())[:5]:
        print(f"      {c}× — {f}")
else:
    max_seen = max(reuse_count.values()) if reuse_count else 0
    print(f"    ✓ No violations. Max reuse seen: {max_seen}×")

# ── 13d. SNR distribution ────────────────────────────────────────────────
print("\n▸ SNR band distribution:")
band_counts = df["snr_band"].value_counts()
for band in SNR_BANDS:
    lbl = band["label"]
    cnt = band_counts.get(lbl, 0)
    frac = cnt / len(df)
    target = band["prob"]
    ok = abs(frac - target) < 0.03
    print(f"    {lbl:<12}: {cnt:>5}  ({frac:.1%})  target={target:.0%}  {'✓' if ok else '~'}")

# ── 13e. stem1/stem2 same class check ────────────────────────────────────
print("\n▸ stem1/stem2 same-class check:")
same_class = df[df["stem1_class"] == df["stem2_class"]]
if len(same_class) > 0:
    print(f"    ✗ {len(same_class)} clips have stem1_class == stem2_class!")
else:
    print("    ✓ No clips have stem1/stem2 same class.")

# ── 13f. Mixed-label check ────────────────────────────────────────────────
print("\n▸ Mixed-label check:")
def label_of_class(cls):
    if cls in COOKING_FG_CLASSES:   return 1
    if cls in NONCOOKING_FG_CLASSES: return 0
    return None  # background

df["_s1_lbl"] = df["stem1_class"].apply(label_of_class)
df["_s2_lbl"] = df["stem2_class"].apply(label_of_class)
mixed = df[df["_s1_lbl"] != df["_s2_lbl"]]
if len(mixed) > 0:
    print(f"    ✗ {len(mixed)} clips have mixed stem labels!")
else:
    print("    ✓ No mixed-label clips.")
df.drop(columns=["_s1_lbl", "_s2_lbl"], inplace=True)

# ── 13g. num_stems distribution ──────────────────────────────────────────
print("\n▸ num_stems distribution:")
stem_counts = df["num_stems"].value_counts().sort_index()
for ns, cnt in stem_counts.items():
    frac = cnt / len(df)
    target = STEM_PROBS.get(ns, 0)
    ok = abs(frac - target) < 0.04
    print(f"    {ns}-stem: {cnt:>5}  ({frac:.1%})  target={target:.0%}  {'✓' if ok else '~'}")

print("\n" + "═" * 60)
print("  generation_plan.csv is ready for Phase 2 (mixing).")
print("═" * 60 + "\n")

# ─────────────────────────────────────────────────────────────────────────────
# 14. WRITE GENERATION PLAN SHEET INTO pipeline_report.xlsx
# ─────────────────────────────────────────────────────────────────────────────
print("[Phase 1] Writing sheet to pipeline_report.xlsx ...")

SHEET_NAME = "Generation Plan"

def style_header_row(ws, num_cols: int, fill_hex: str = "1F4E79"):
    fill   = PatternFill("solid", fgColor=fill_hex)
    font   = Font(bold=True, color="FFFFFF", size=9)
    align  = Alignment(horizontal="center", vertical="center", wrap_text=True)
    thin   = Side(style="thin", color="FFFFFF")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    for cell in ws[1]:
        cell.fill   = fill
        cell.font   = font
        cell.alignment = align
        cell.border = border

def auto_col_width(ws, max_w: int = 40):
    for col in ws.columns:
        lengths = [len(str(c.value)) if c.value is not None else 0 for c in col]
        ws.column_dimensions[col[0].column_letter].width = min(max(lengths) + 2, max_w)

if REPORT_XLS.exists():
    wb = load_workbook(REPORT_XLS)
else:
    from openpyxl import Workbook
    wb = Workbook()
    if "Sheet" in wb.sheetnames:
        del wb["Sheet"]

# Remove old sheet if exists
if SHEET_NAME in wb.sheetnames:
    del wb[SHEET_NAME]

ws = wb.create_sheet(SHEET_NAME)

# Write header
ws.append(list(df_out.columns))

# Write data rows (first 2500 rows — all of them)
for _, row in df_out.iterrows():
    ws.append([
        row["file_id"], row["label"], row["num_stems"], row["split"], row["SNR_dB"],
        row["stem1_class"], row["stem1_file"], row["stem1_role"], row["stem1_gain_dB"],
        row["stem2_class"], row["stem2_file"], row["stem2_role"], row["stem2_gain_dB"],
        row["stem3_class"], row["stem3_file"], row["stem3_role"], row["stem3_gain_dB"],
        row["stem4_class"], row["stem4_file"], row["stem4_role"], row["stem4_gain_dB"],
        row["mixing_method"], row["status"],
    ])

style_header_row(ws, len(df_out.columns))
auto_col_width(ws)
ws.freeze_panes = "A2"
ws.row_dimensions[1].height = 28

wb.save(REPORT_XLS)
print(f"  Saved sheet '{SHEET_NAME}' → {REPORT_XLS}\n")

print("[Phase 1] COMPLETE ✓")
print(f"  Output: {PLAN_CSV}")
print(f"  Rows:   {len(df_out)}")
print(f"  Next:   Run phase2_mixing.py\n")

In [ ]:
"""
phase1_generation_plan_v2.py
Samsung PRISM — Kitchen Audio Dataset v2
Phase 1: Generation Plan  (v2 — fixes non-cooking reuse bottleneck)

Change from v1:
  MAX_REUSE is now per-category:
    cooking FG    → 3  (883 files, hits 1250 comfortably)
    noncooking FG → 6  (438 files, needs higher reuse to reach 1250)
    background    → 3  (unchanged — large pool)

Run on Google Colab CPU.
"""

# ─────────────────────────────────────────────────────────────────────────────
# 0. IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import os, sys, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from collections import defaultdict
from pathlib import Path

try:
    from openpyxl import load_workbook
    from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
except ImportError:
    os.system("pip install openpyxl -q")
    from openpyxl import load_workbook
    from openpyxl.styles import PatternFill, Font, Alignment, Border, Side

# ─────────────────────────────────────────────────────────────────────────────
# 1. MOUNT GOOGLE DRIVE
# ─────────────────────────────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
    print("[OK] Google Drive mounted.")
except ModuleNotFoundError:
    DRIVE_ROOT = os.path.expanduser("~/kitpri_dev")
    print(f"[INFO] Not in Colab — using local path: {DRIVE_ROOT}")

# ─────────────────────────────────────────────────────────────────────────────
# 2. PATHS
# ─────────────────────────────────────────────────────────────────────────────
META_DIR   = Path(DRIVE_ROOT) / "kitpri" / "kitpri_v2" / "metadata"
INV_CSV    = META_DIR / "inventory.csv"
PLAN_CSV   = META_DIR / "generation_plan.csv"
REPORT_XLS = META_DIR / "pipeline_report.xlsx"

META_DIR.mkdir(parents=True, exist_ok=True)
assert INV_CSV.exists(), f"[ERROR] inventory.csv not found at {INV_CSV}"
print(f"[OK] inventory.csv found: {INV_CSV}")

# ─────────────────────────────────────────────────────────────────────────────
# 3. CLASS DEFINITIONS (locked)
# ─────────────────────────────────────────────────────────────────────────────
COOKING_FG_CLASSES = [
    "frying", "chopping", "boiling", "blender", "stove",
    "dishes", "microwave", "pressure_cooker", "water_tap", "stirring",
]
NONCOOKING_FG_CLASSES = [
    "speech", "TV_audio", "music", "keyboard_typing", "dog_barking",
    "phone_ringing", "door_knock", "vacuum_cleaner", "footsteps",
]
BACKGROUND_CLASSES = [
    "room_ambience", "fan_ac", "musan_noise",
    "street_noise", "rain", "noncooking_other",
]
ALL_FG_CLASSES = COOKING_FG_CLASSES + NONCOOKING_FG_CLASSES
ALL_CLASSES    = ALL_FG_CLASSES + BACKGROUND_CLASSES

# ─────────────────────────────────────────────────────────────────────────────
# 4. DATASET DESIGN CONSTANTS
# ─────────────────────────────────────────────────────────────────────────────
TOTAL_CLIPS      = 2500
COOKING_CLIPS    = 1250
NONCOOKING_CLIPS = 1250
SEED             = 42

# Per-category reuse limits
MAX_REUSE_BY_CATEGORY = {
    "cooking_fg":    3,   # 883 files → 2649 tokens → 1324 theoretical clips ✓
    "noncooking_fg": 6,   # 438 files → 2628 tokens → 1314 theoretical clips ✓
    "background":    3,   # large pool, no issue
}

STEM_PROBS = {3: 0.80, 4: 0.20}

SNR_BANDS = [
    {"label": "very_hard", "low": -5,  "high":  0, "prob": 0.10},
    {"label": "hard",      "low":  0,  "high":  5, "prob": 0.20},
    {"label": "medium",    "low":  5,  "high": 10, "prob": 0.30},
    {"label": "easy",      "low": 10,  "high": 20, "prob": 0.30},
    {"label": "very_easy", "low": 20,  "high": 30, "prob": 0.10},
]

SPLIT_RATIOS = {"train": 0.70, "val": 0.15, "test": 0.15}

# ─────────────────────────────────────────────────────────────────────────────
# 5. LOAD INVENTORY
# ─────────────────────────────────────────────────────────────────────────────
print("\n[Phase 1] Loading inventory.csv ...")
inv = pd.read_csv(INV_CSV)
inv.columns = [c.strip() for c in inv.columns]

assert {"file_path", "audio_class"}.issubset(set(inv.columns)), \
    "[ERROR] inventory.csv must have 'file_path' and 'audio_class' columns"

print(f"  Total rows in inventory: {len(inv)}")

file_pool: dict[str, list[str]] = defaultdict(list)
for _, row in inv.iterrows():
    cls  = str(row["audio_class"]).strip()
    path = str(row["file_path"]).strip()
    if cls in ALL_CLASSES:
        file_pool[cls].append(path)

print("\n  File pool sizes per class:")
for cls in ALL_CLASSES:
    n = len(file_pool.get(cls, []))
    cat = ("cooking_fg" if cls in COOKING_FG_CLASSES
           else "noncooking_fg" if cls in NONCOOKING_FG_CLASSES
           else "background")
    limit = MAX_REUSE_BY_CATEGORY[cat]
    flag = "  [EMPTY!]" if n == 0 else ""
    print(f"    {cls:<22}: {n:>4} files  reuse_limit={limit}  max_clips={n*limit}{flag}")

empty = [c for c in ALL_CLASSES if not file_pool.get(c)]
if empty:
    print(f"\n  [WARNING] Empty classes: {empty}")

# ─────────────────────────────────────────────────────────────────────────────
# 6. REUSE TRACKER & FILE SAMPLER
# ─────────────────────────────────────────────────────────────────────────────
reuse_count: dict[str, int] = defaultdict(int)

def category_of(cls: str) -> str:
    if cls in COOKING_FG_CLASSES:    return "cooking_fg"
    if cls in NONCOOKING_FG_CLASSES: return "noncooking_fg"
    return "background"

def max_reuse_for(cls: str) -> int:
    return MAX_REUSE_BY_CATEGORY[category_of(cls)]

def available_files(cls: str) -> list[str]:
    limit = max_reuse_for(cls)
    return [f for f in file_pool.get(cls, []) if reuse_count[f] < limit]

def pick_file(cls: str, rng: random.Random, exclude: list[str] = None) -> str | None:
    candidates = available_files(cls)
    if exclude:
        candidates = [f for f in candidates if f not in exclude]
    if not candidates:
        return None
    chosen = rng.choice(candidates)
    reuse_count[chosen] += 1
    return chosen

# ─────────────────────────────────────────────────────────────────────────────
# 7. SNR SAMPLER
# ─────────────────────────────────────────────────────────────────────────────
def pick_snr(rng: random.Random) -> tuple[float, str]:
    r = rng.random()
    cumulative = 0.0
    for band in SNR_BANDS:
        cumulative += band["prob"]
        if r <= cumulative:
            return round(rng.uniform(band["low"], band["high"]), 2), band["label"]
    return round(rng.uniform(SNR_BANDS[-1]["low"], SNR_BANDS[-1]["high"]), 2), SNR_BANDS[-1]["label"]

# ─────────────────────────────────────────────────────────────────────────────
# 8. GAIN CALCULATOR
# ─────────────────────────────────────────────────────────────────────────────
def compute_gains(snr_db: float, rng: random.Random, num_stems: int) -> dict:
    g1 = round(rng.uniform(-3.0, 0.0), 2)
    g2 = round(g1 - rng.uniform(3.0, 9.0), 2)
    g3 = round(g1 - snr_db, 2)
    g4 = round(g3 - rng.uniform(12.0, 18.0), 2) if num_stems == 4 else None
    return {"stem1": g1, "stem2": g2, "stem3": g3, "stem4": g4}

# ─────────────────────────────────────────────────────────────────────────────
# 9. STRATIFIED SPLIT
# ─────────────────────────────────────────────────────────────────────────────
def assign_splits(df: pd.DataFrame, rng: random.Random) -> pd.Series:
    splits = pd.Series(index=df.index, dtype=str)
    for _, grp in df.groupby(["label", "snr_band", "num_stems"]):
        indices = grp.index.tolist()
        rng.shuffle(indices)
        n = len(indices)
        n_val  = max(1, round(n * SPLIT_RATIOS["val"]))
        n_test = max(1, round(n * SPLIT_RATIOS["test"]))
        splits[indices[:n_val]]              = "val"
        splits[indices[n_val:n_val+n_test]]  = "test"
        splits[indices[n_val+n_test:]]       = "train"
    return splits

# ─────────────────────────────────────────────────────────────────────────────
# 10. GENERATE CLIP RECIPES
# ─────────────────────────────────────────────────────────────────────────────
print("\n[Phase 1] Generating clip recipes ...")
rng = random.Random(SEED)
np.random.seed(SEED)

records = []
skipped = 0

def make_clips(label: int, fg_classes: list[str], count: int):
    global skipped
    made, attempts = 0, 0
    max_attempts = count * 20

    while made < count and attempts < max_attempts:
        attempts += 1

        num_stems = 3 if rng.random() < STEM_PROBS[3] else 4

        # stem1
        s1_class = rng.choice(fg_classes)
        s1_file  = pick_file(s1_class, rng)
        if s1_file is None:
            skipped += 1; continue

        # stem2 — different class, same category
        other_fg = [c for c in fg_classes if c != s1_class]
        if not other_fg:
            skipped += 1; continue
        rng.shuffle(other_fg)
        s2_class = s2_file = None
        for cand in other_fg:
            f = pick_file(cand, rng, exclude=[s1_file])
            if f:
                s2_class, s2_file = cand, f
                break
        if s2_file is None:
            skipped += 1; continue

        # stem3 — background
        bg_avail = [c for c in BACKGROUND_CLASSES if available_files(c)]
        if not bg_avail:
            skipped += 1; continue
        s3_class = rng.choice(bg_avail)
        s3_file  = pick_file(s3_class, rng)
        if s3_file is None:
            skipped += 1; continue

        # stem4 — different background class (downgrade to 3-stem if unavailable)
        s4_class = s4_file = None
        if num_stems == 4:
            other_bg = [c for c in BACKGROUND_CLASSES if c != s3_class and available_files(c)]
            if other_bg:
                s4_class = rng.choice(other_bg)
                s4_file  = pick_file(s4_class, rng, exclude=[s3_file])
            if s4_file is None:
                num_stems = 3

        snr_db, snr_band = pick_snr(rng)
        gains = compute_gains(snr_db, rng, num_stems)

        lbl_prefix = "c" if label == 1 else "n"
        file_id = f"{lbl_prefix}_{made+1:05d}"

        records.append({
            "file_id":       file_id,
            "label":         label,
            "num_stems":     num_stems,
            "split":         "",
            "SNR_dB":        snr_db,
            "snr_band":      snr_band,
            "stem1_class":   s1_class,
            "stem1_file":    s1_file,
            "stem1_role":    "fg_primary",
            "stem1_gain_dB": gains["stem1"],
            "stem2_class":   s2_class,
            "stem2_file":    s2_file,
            "stem2_role":    "fg_secondary",
            "stem2_gain_dB": gains["stem2"],
            "stem3_class":   s3_class,
            "stem3_file":    s3_file,
            "stem3_role":    "bg_primary",
            "stem3_gain_dB": gains["stem3"],
            "stem4_class":   s4_class   if num_stems == 4 else "",
            "stem4_file":    s4_file    if num_stems == 4 else "",
            "stem4_role":    "bg_secondary" if num_stems == 4 else "",
            "stem4_gain_dB": gains["stem4"] if num_stems == 4 else "",
            "mixing_method": "librosa",
            "status":        "pending",
        })
        made += 1

    if made < count:
        print(f"  [WARNING] Only generated {made}/{count} clips for label={label}")

print("  Generating cooking clips   (label=1) ...")
make_clips(label=1, fg_classes=COOKING_FG_CLASSES,    count=COOKING_CLIPS)
print("  Generating noncooking clips (label=0) ...")
make_clips(label=0, fg_classes=NONCOOKING_FG_CLASSES, count=NONCOOKING_CLIPS)

df = pd.DataFrame(records)
print(f"\n  Total recipes generated: {len(df)}  (skipped: {skipped})")

# ─────────────────────────────────────────────────────────────────────────────
# 11. ASSIGN SPLITS
# ─────────────────────────────────────────────────────────────────────────────
print("\n[Phase 1] Assigning stratified splits ...")
split_rng = random.Random(SEED + 1)
df["split"] = assign_splits(df, split_rng)

# ─────────────────────────────────────────────────────────────────────────────
# 12. SAVE generation_plan.csv
# ─────────────────────────────────────────────────────────────────────────────
OUTPUT_COLS = [
    "file_id", "label", "num_stems", "split", "SNR_dB",
    "stem1_class", "stem1_file", "stem1_role", "stem1_gain_dB",
    "stem2_class", "stem2_file", "stem2_role", "stem2_gain_dB",
    "stem3_class", "stem3_file", "stem3_role", "stem3_gain_dB",
    "stem4_class", "stem4_file", "stem4_role", "stem4_gain_dB",
    "mixing_method", "status",
]
df_out = df[OUTPUT_COLS]
df_out.to_csv(PLAN_CSV, index=False)
print(f"  Saved: {PLAN_CSV}  ({len(df_out)} rows)")

# ─────────────────────────────────────────────────────────────────────────────
# 13. VALIDATION REPORT
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "═" * 62)
print("  PHASE 1 VALIDATION REPORT  (v2 — per-category reuse limits)")
print("═" * 62)

print("\n▸ Clip counts by label:")
label_counts = df["label"].value_counts().sort_index()
for lbl, cnt in label_counts.items():
    name   = "cooking" if lbl == 1 else "non-cooking"
    target = COOKING_CLIPS if lbl == 1 else NONCOOKING_CLIPS
    status = "✓" if cnt == target else f"✗ (target={target})"
    print(f"    label={lbl} ({name}): {cnt}  {status}")
total_ok = len(df) == TOTAL_CLIPS
print(f"    TOTAL: {len(df)} / {TOTAL_CLIPS}  {'✓' if total_ok else '✗'}")

print("\n▸ Split distribution:")
split_counts = df["split"].value_counts()
for split in ["train", "val", "test"]:
    cnt  = split_counts.get(split, 0)
    frac = cnt / len(df)
    target = SPLIT_RATIOS[split]
    ok   = abs(frac - target) < 0.03
    print(f"    {split:<6}: {cnt:>5}  ({frac:.1%})  target={target:.0%}  {'✓' if ok else '~'}")

print("\n▸ File reuse limit check (per-category):")
for category, limit in MAX_REUSE_BY_CATEGORY.items():
    classes_in_cat = (COOKING_FG_CLASSES if category == "cooking_fg"
                      else NONCOOKING_FG_CLASSES if category == "noncooking_fg"
                      else BACKGROUND_CLASSES)
    cat_files = [f for cls in classes_in_cat for f in file_pool.get(cls, [])]
    violations = {f: reuse_count[f] for f in cat_files if reuse_count[f] > limit}
    max_seen   = max((reuse_count[f] for f in cat_files), default=0)
    status = f"✓ max_seen={max_seen}" if not violations else f"✗ {len(violations)} violations"
    print(f"    {category:<15} limit={limit}  {status}")

print("\n▸ SNR band distribution:")
band_counts = df["snr_band"].value_counts()
for band in SNR_BANDS:
    lbl    = band["label"]
    cnt    = band_counts.get(lbl, 0)
    frac   = cnt / len(df)
    target = band["prob"]
    ok     = abs(frac - target) < 0.03
    print(f"    {lbl:<12}: {cnt:>5}  ({frac:.1%})  target={target:.0%}  {'✓' if ok else '~'}")

print("\n▸ stem1/stem2 same-class check:")
same = df[df["stem1_class"] == df["stem2_class"]]
print(f"    {'✓ No violations.' if len(same)==0 else f'✗ {len(same)} violations!'}")

print("\n▸ Mixed-label check:")
df["_s1l"] = df["stem1_class"].apply(
    lambda c: 1 if c in COOKING_FG_CLASSES else (0 if c in NONCOOKING_FG_CLASSES else None))
df["_s2l"] = df["stem2_class"].apply(
    lambda c: 1 if c in COOKING_FG_CLASSES else (0 if c in NONCOOKING_FG_CLASSES else None))
mixed = df[df["_s1l"] != df["_s2l"]]
print(f"    {'✓ No mixed-label clips.' if len(mixed)==0 else f'✗ {len(mixed)} mixed-label clips!'}")
df.drop(columns=["_s1l", "_s2l"], inplace=True)

print("\n▸ num_stems distribution:")
stem_counts = df["num_stems"].value_counts().sort_index()
for ns, cnt in stem_counts.items():
    frac   = cnt / len(df)
    target = STEM_PROBS.get(ns, 0)
    ok     = abs(frac - target) < 0.04
    print(f"    {ns}-stem: {cnt:>5}  ({frac:.1%})  target={target:.0%}  {'✓' if ok else '~'}")

print("\n▸ Reuse distribution across all used files:")
used = {f: c for f, c in reuse_count.items() if c > 0}
from collections import Counter
dist = Counter(used.values())
for times in sorted(dist):
    print(f"    used {times}×: {dist[times]} files")

print("\n" + "═" * 62)
print("  generation_plan.csv is ready for Phase 2 (mixing).")
print("═" * 62 + "\n")

# ─────────────────────────────────────────────────────────────────────────────
# 14. WRITE SHEET INTO pipeline_report.xlsx
# ─────────────────────────────────────────────────────────────────────────────
print("[Phase 1] Writing sheet to pipeline_report.xlsx ...")

def style_header_row(ws, fill_hex: str = "1F4E79"):
    fill   = PatternFill("solid", fgColor=fill_hex)
    font   = Font(bold=True, color="FFFFFF", size=9)
    align  = Alignment(horizontal="center", vertical="center", wrap_text=True)
    thin   = Side(style="thin", color="FFFFFF")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    for cell in ws[1]:
        cell.fill      = fill
        cell.font      = font
        cell.alignment = align
        cell.border    = border

def auto_col_width(ws, max_w: int = 40):
    for col in ws.columns:
        lengths = [len(str(c.value)) if c.value is not None else 0 for c in col]
        ws.column_dimensions[col[0].column_letter].width = min(max(lengths) + 2, max_w)

SHEET_NAME = "Generation Plan"
if REPORT_XLS.exists():
    wb = load_workbook(REPORT_XLS)
else:
    from openpyxl import Workbook
    wb = Workbook()
    if "Sheet" in wb.sheetnames:
        del wb["Sheet"]

if SHEET_NAME in wb.sheetnames:
    del wb[SHEET_NAME]
ws = wb.create_sheet(SHEET_NAME)

ws.append(list(df_out.columns))
for _, row in df_out.iterrows():
    ws.append([row[c] for c in df_out.columns])

style_header_row(ws)
auto_col_width(ws)
ws.freeze_panes = "A2"
ws.row_dimensions[1].height = 28

wb.save(REPORT_XLS)
print(f"  Saved sheet '{SHEET_NAME}' → {REPORT_XLS}\n")

print("[Phase 1 v2] COMPLETE ✓")
print(f"  Output: {PLAN_CSV}")
print(f"  Rows:   {len(df_out)}")
print(f"  Next:   Run phase2_mixing.py\n")

In [ ]:
"""
phase1_generation_plan_v3.py
Samsung PRISM — Kitchen Audio Dataset v2
Phase 1: Generation Plan  (v3 — fixes noncooking shortfall + 4-stem distribution)

Changes from v2:
  noncooking_fg reuse: 6 → 7  (438 files × 7 = 3066 tokens → 1533 max clips)
  background reuse:    3 → 4  (small bg classes were exhausting, killing 4-stem)
  4-stem logic:  retry with different stem3 up to 5× before downgrading to 3-stem

Run on Google Colab CPU.
"""

# ─────────────────────────────────────────────────────────────────────────────
# 0. IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import os, sys, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from collections import defaultdict
from pathlib import Path

try:
    from openpyxl import load_workbook
    from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
except ImportError:
    os.system("pip install openpyxl -q")
    from openpyxl import load_workbook
    from openpyxl.styles import PatternFill, Font, Alignment, Border, Side

# ─────────────────────────────────────────────────────────────────────────────
# 1. MOUNT GOOGLE DRIVE
# ─────────────────────────────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
    print("[OK] Google Drive mounted.")
except ModuleNotFoundError:
    DRIVE_ROOT = os.path.expanduser("~/kitpri_dev")
    print(f"[INFO] Not in Colab — using local path: {DRIVE_ROOT}")

# ─────────────────────────────────────────────────────────────────────────────
# 2. PATHS
# ─────────────────────────────────────────────────────────────────────────────
META_DIR   = Path(DRIVE_ROOT) / "kitpri" / "kitpri_v2" / "metadata"
INV_CSV    = META_DIR / "inventory.csv"
PLAN_CSV   = META_DIR / "generation_plan.csv"
REPORT_XLS = META_DIR / "pipeline_report.xlsx"

META_DIR.mkdir(parents=True, exist_ok=True)
assert INV_CSV.exists(), f"[ERROR] inventory.csv not found at {INV_CSV}"
print(f"[OK] inventory.csv found: {INV_CSV}")

# ─────────────────────────────────────────────────────────────────────────────
# 3. CLASS DEFINITIONS (locked)
# ─────────────────────────────────────────────────────────────────────────────
COOKING_FG_CLASSES = [
    "frying", "chopping", "boiling", "blender", "stove",
    "dishes", "microwave", "pressure_cooker", "water_tap", "stirring",
]
NONCOOKING_FG_CLASSES = [
    "speech", "TV_audio", "music", "keyboard_typing", "dog_barking",
    "phone_ringing", "door_knock", "vacuum_cleaner", "footsteps",
]
BACKGROUND_CLASSES = [
    "room_ambience", "fan_ac", "musan_noise",
    "street_noise", "rain", "noncooking_other",
]
ALL_FG_CLASSES = COOKING_FG_CLASSES + NONCOOKING_FG_CLASSES
ALL_CLASSES    = ALL_FG_CLASSES + BACKGROUND_CLASSES

# ─────────────────────────────────────────────────────────────────────────────
# 4. DATASET DESIGN CONSTANTS
# ─────────────────────────────────────────────────────────────────────────────
TOTAL_CLIPS      = 2500
COOKING_CLIPS    = 1250
NONCOOKING_CLIPS = 1250
SEED             = 42

# Per-category reuse limits
MAX_REUSE_BY_CATEGORY = {
    "cooking_fg":    3,   # 883 files → 2649 tokens → 1324 theoretical clips ✓
    "noncooking_fg": 7,   # 438 files → 3066 tokens → 1533 theoretical clips ✓ (was 6→1314, 44 short)
    "background":    4,   # raised from 3 — small classes (rain=40, fan_ac=43) exhausted at 3×
                          # causing 4-stem clips to silently downgrade; 4× gives safe headroom
}

STEM_PROBS = {3: 0.80, 4: 0.20}

SNR_BANDS = [
    {"label": "very_hard", "low": -5,  "high":  0, "prob": 0.10},
    {"label": "hard",      "low":  0,  "high":  5, "prob": 0.20},
    {"label": "medium",    "low":  5,  "high": 10, "prob": 0.30},
    {"label": "easy",      "low": 10,  "high": 20, "prob": 0.30},
    {"label": "very_easy", "low": 20,  "high": 30, "prob": 0.10},
]

SPLIT_RATIOS = {"train": 0.70, "val": 0.15, "test": 0.15}

# ─────────────────────────────────────────────────────────────────────────────
# 5. LOAD INVENTORY
# ─────────────────────────────────────────────────────────────────────────────
print("\n[Phase 1] Loading inventory.csv ...")
inv = pd.read_csv(INV_CSV)
inv.columns = [c.strip() for c in inv.columns]

assert {"file_path", "audio_class"}.issubset(set(inv.columns)), \
    "[ERROR] inventory.csv must have 'file_path' and 'audio_class' columns"

print(f"  Total rows in inventory: {len(inv)}")

file_pool: dict[str, list[str]] = defaultdict(list)
for _, row in inv.iterrows():
    cls  = str(row["audio_class"]).strip()
    path = str(row["file_path"]).strip()
    if cls in ALL_CLASSES:
        file_pool[cls].append(path)

print("\n  File pool sizes per class:")
for cls in ALL_CLASSES:
    n = len(file_pool.get(cls, []))
    cat = ("cooking_fg" if cls in COOKING_FG_CLASSES
           else "noncooking_fg" if cls in NONCOOKING_FG_CLASSES
           else "background")
    limit = MAX_REUSE_BY_CATEGORY[cat]
    flag = "  [EMPTY!]" if n == 0 else ""
    print(f"    {cls:<22}: {n:>4} files  reuse_limit={limit}  max_clips={n*limit}{flag}")

empty = [c for c in ALL_CLASSES if not file_pool.get(c)]
if empty:
    print(f"\n  [WARNING] Empty classes: {empty}")

# ─────────────────────────────────────────────────────────────────────────────
# 6. REUSE TRACKER & FILE SAMPLER
# ─────────────────────────────────────────────────────────────────────────────
reuse_count: dict[str, int] = defaultdict(int)

def category_of(cls: str) -> str:
    if cls in COOKING_FG_CLASSES:    return "cooking_fg"
    if cls in NONCOOKING_FG_CLASSES: return "noncooking_fg"
    return "background"

def max_reuse_for(cls: str) -> int:
    return MAX_REUSE_BY_CATEGORY[category_of(cls)]

def available_files(cls: str) -> list[str]:
    limit = max_reuse_for(cls)
    return [f for f in file_pool.get(cls, []) if reuse_count[f] < limit]

def pick_file(cls: str, rng: random.Random, exclude: list[str] = None) -> str | None:
    candidates = available_files(cls)
    if exclude:
        candidates = [f for f in candidates if f not in exclude]
    if not candidates:
        return None
    chosen = rng.choice(candidates)
    reuse_count[chosen] += 1
    return chosen

# ─────────────────────────────────────────────────────────────────────────────
# 7. SNR SAMPLER
# ─────────────────────────────────────────────────────────────────────────────
def pick_snr(rng: random.Random) -> tuple[float, str]:
    r = rng.random()
    cumulative = 0.0
    for band in SNR_BANDS:
        cumulative += band["prob"]
        if r <= cumulative:
            return round(rng.uniform(band["low"], band["high"]), 2), band["label"]
    return round(rng.uniform(SNR_BANDS[-1]["low"], SNR_BANDS[-1]["high"]), 2), SNR_BANDS[-1]["label"]

# ─────────────────────────────────────────────────────────────────────────────
# 8. GAIN CALCULATOR
# ─────────────────────────────────────────────────────────────────────────────
def compute_gains(snr_db: float, rng: random.Random, num_stems: int) -> dict:
    g1 = round(rng.uniform(-3.0, 0.0), 2)
    g2 = round(g1 - rng.uniform(3.0, 9.0), 2)
    g3 = round(g1 - snr_db, 2)
    g4 = round(g3 - rng.uniform(12.0, 18.0), 2) if num_stems == 4 else None
    return {"stem1": g1, "stem2": g2, "stem3": g3, "stem4": g4}

# ─────────────────────────────────────────────────────────────────────────────
# 9. STRATIFIED SPLIT
# ─────────────────────────────────────────────────────────────────────────────
def assign_splits(df: pd.DataFrame, rng: random.Random) -> pd.Series:
    splits = pd.Series(index=df.index, dtype=str)
    for _, grp in df.groupby(["label", "snr_band", "num_stems"]):
        indices = grp.index.tolist()
        rng.shuffle(indices)
        n = len(indices)
        n_val  = max(1, round(n * SPLIT_RATIOS["val"]))
        n_test = max(1, round(n * SPLIT_RATIOS["test"]))
        splits[indices[:n_val]]              = "val"
        splits[indices[n_val:n_val+n_test]]  = "test"
        splits[indices[n_val+n_test:]]       = "train"
    return splits

# ─────────────────────────────────────────────────────────────────────────────
# 10. GENERATE CLIP RECIPES
# ─────────────────────────────────────────────────────────────────────────────
print("\n[Phase 1] Generating clip recipes ...")
rng = random.Random(SEED)
np.random.seed(SEED)

records = []
skipped = 0

def make_clips(label: int, fg_classes: list[str], count: int):
    global skipped
    made, attempts = 0, 0
    max_attempts = count * 20

    while made < count and attempts < max_attempts:
        attempts += 1

        num_stems = 3 if rng.random() < STEM_PROBS[3] else 4

        # stem1
        s1_class = rng.choice(fg_classes)
        s1_file  = pick_file(s1_class, rng)
        if s1_file is None:
            skipped += 1; continue

        # stem2 — different class, same category
        other_fg = [c for c in fg_classes if c != s1_class]
        if not other_fg:
            skipped += 1; continue
        rng.shuffle(other_fg)
        s2_class = s2_file = None
        for cand in other_fg:
            f = pick_file(cand, rng, exclude=[s1_file])
            if f:
                s2_class, s2_file = cand, f
                break
        if s2_file is None:
            skipped += 1; continue

        # stem3 — background
        bg_avail = [c for c in BACKGROUND_CLASSES if available_files(c)]
        if not bg_avail:
            skipped += 1; continue
        s3_class = rng.choice(bg_avail)
        s3_file  = pick_file(s3_class, rng)
        if s3_file is None:
            skipped += 1; continue

        # stem4 — different background class
        # For 4-stem clips: retry stem3+stem4 pair up to 5× before downgrading.
        # This prevents small bg classes exhausting and silently killing 4-stem distribution.
        s4_class = s4_file = None
        if num_stems == 4:
            found_pair = False
            for _retry in range(5):
                other_bg = [c for c in BACKGROUND_CLASSES if c != s3_class and available_files(c)]
                if other_bg:
                    s4_class_try = rng.choice(other_bg)
                    s4_file_try  = pick_file(s4_class_try, rng, exclude=[s3_file])
                    if s4_file_try:
                        s4_class, s4_file = s4_class_try, s4_file_try
                        found_pair = True
                        break
                # stem4 unavailable with current stem3 — try a different stem3
                reuse_count[s3_file] -= 1  # undo the stem3 pick
                bg_avail2 = [c for c in BACKGROUND_CLASSES if available_files(c)]
                if not bg_avail2:
                    break
                s3_class = rng.choice(bg_avail2)
                s3_file  = pick_file(s3_class, rng)
                if s3_file is None:
                    break
            if not found_pair:
                num_stems = 3  # last resort downgrade only after 5 retries

        snr_db, snr_band = pick_snr(rng)
        gains = compute_gains(snr_db, rng, num_stems)

        lbl_prefix = "c" if label == 1 else "n"
        file_id = f"{lbl_prefix}_{made+1:05d}"

        records.append({
            "file_id":       file_id,
            "label":         label,
            "num_stems":     num_stems,
            "split":         "",
            "SNR_dB":        snr_db,
            "snr_band":      snr_band,
            "stem1_class":   s1_class,
            "stem1_file":    s1_file,
            "stem1_role":    "fg_primary",
            "stem1_gain_dB": gains["stem1"],
            "stem2_class":   s2_class,
            "stem2_file":    s2_file,
            "stem2_role":    "fg_secondary",
            "stem2_gain_dB": gains["stem2"],
            "stem3_class":   s3_class,
            "stem3_file":    s3_file,
            "stem3_role":    "bg_primary",
            "stem3_gain_dB": gains["stem3"],
            "stem4_class":   s4_class   if num_stems == 4 else "",
            "stem4_file":    s4_file    if num_stems == 4 else "",
            "stem4_role":    "bg_secondary" if num_stems == 4 else "",
            "stem4_gain_dB": gains["stem4"] if num_stems == 4 else "",
            "mixing_method": "librosa",
            "status":        "pending",
        })
        made += 1

    if made < count:
        print(f"  [WARNING] Only generated {made}/{count} clips for label={label}")

print("  Generating cooking clips   (label=1) ...")
make_clips(label=1, fg_classes=COOKING_FG_CLASSES,    count=COOKING_CLIPS)
print("  Generating noncooking clips (label=0) ...")
make_clips(label=0, fg_classes=NONCOOKING_FG_CLASSES, count=NONCOOKING_CLIPS)

df = pd.DataFrame(records)
print(f"\n  Total recipes generated: {len(df)}  (skipped: {skipped})")

# ─────────────────────────────────────────────────────────────────────────────
# 11. ASSIGN SPLITS
# ─────────────────────────────────────────────────────────────────────────────
print("\n[Phase 1] Assigning stratified splits ...")
split_rng = random.Random(SEED + 1)
df["split"] = assign_splits(df, split_rng)

# ─────────────────────────────────────────────────────────────────────────────
# 12. SAVE generation_plan.csv
# ─────────────────────────────────────────────────────────────────────────────
OUTPUT_COLS = [
    "file_id", "label", "num_stems", "split", "SNR_dB",
    "stem1_class", "stem1_file", "stem1_role", "stem1_gain_dB",
    "stem2_class", "stem2_file", "stem2_role", "stem2_gain_dB",
    "stem3_class", "stem3_file", "stem3_role", "stem3_gain_dB",
    "stem4_class", "stem4_file", "stem4_role", "stem4_gain_dB",
    "mixing_method", "status",
]
df_out = df[OUTPUT_COLS]
df_out.to_csv(PLAN_CSV, index=False)
print(f"  Saved: {PLAN_CSV}  ({len(df_out)} rows)")

# ─────────────────────────────────────────────────────────────────────────────
# 13. VALIDATION REPORT
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "═" * 62)
print("  PHASE 1 VALIDATION REPORT  (v3 — per-category reuse limits + 4-stem retry)")
print("═" * 62)

print("\n▸ Clip counts by label:")
label_counts = df["label"].value_counts().sort_index()
for lbl, cnt in label_counts.items():
    name   = "cooking" if lbl == 1 else "non-cooking"
    target = COOKING_CLIPS if lbl == 1 else NONCOOKING_CLIPS
    status = "✓" if cnt == target else f"✗ (target={target})"
    print(f"    label={lbl} ({name}): {cnt}  {status}")
total_ok = len(df) == TOTAL_CLIPS
print(f"    TOTAL: {len(df)} / {TOTAL_CLIPS}  {'✓' if total_ok else '✗'}")

print("\n▸ Split distribution:")
split_counts = df["split"].value_counts()
for split in ["train", "val", "test"]:
    cnt  = split_counts.get(split, 0)
    frac = cnt / len(df)
    target = SPLIT_RATIOS[split]
    ok   = abs(frac - target) < 0.03
    print(f"    {split:<6}: {cnt:>5}  ({frac:.1%})  target={target:.0%}  {'✓' if ok else '~'}")

print("\n▸ File reuse limit check (per-category):")
for category, limit in MAX_REUSE_BY_CATEGORY.items():
    classes_in_cat = (COOKING_FG_CLASSES if category == "cooking_fg"
                      else NONCOOKING_FG_CLASSES if category == "noncooking_fg"
                      else BACKGROUND_CLASSES)
    cat_files = [f for cls in classes_in_cat for f in file_pool.get(cls, [])]
    violations = {f: reuse_count[f] for f in cat_files if reuse_count[f] > limit}
    max_seen   = max((reuse_count[f] for f in cat_files), default=0)
    status = f"✓ max_seen={max_seen}" if not violations else f"✗ {len(violations)} violations"
    print(f"    {category:<15} limit={limit}  {status}")

print("\n▸ SNR band distribution:")
band_counts = df["snr_band"].value_counts()
for band in SNR_BANDS:
    lbl    = band["label"]
    cnt    = band_counts.get(lbl, 0)
    frac   = cnt / len(df)
    target = band["prob"]
    ok     = abs(frac - target) < 0.03
    print(f"    {lbl:<12}: {cnt:>5}  ({frac:.1%})  target={target:.0%}  {'✓' if ok else '~'}")

print("\n▸ stem1/stem2 same-class check:")
same = df[df["stem1_class"] == df["stem2_class"]]
print(f"    {'✓ No violations.' if len(same)==0 else f'✗ {len(same)} violations!'}")

print("\n▸ Mixed-label check:")
df["_s1l"] = df["stem1_class"].apply(
    lambda c: 1 if c in COOKING_FG_CLASSES else (0 if c in NONCOOKING_FG_CLASSES else None))
df["_s2l"] = df["stem2_class"].apply(
    lambda c: 1 if c in COOKING_FG_CLASSES else (0 if c in NONCOOKING_FG_CLASSES else None))
mixed = df[df["_s1l"] != df["_s2l"]]
print(f"    {'✓ No mixed-label clips.' if len(mixed)==0 else f'✗ {len(mixed)} mixed-label clips!'}")
df.drop(columns=["_s1l", "_s2l"], inplace=True)

print("\n▸ num_stems distribution:")
stem_counts = df["num_stems"].value_counts().sort_index()
for ns, cnt in stem_counts.items():
    frac   = cnt / len(df)
    target = STEM_PROBS.get(ns, 0)
    ok     = abs(frac - target) < 0.04
    print(f"    {ns}-stem: {cnt:>5}  ({frac:.1%})  target={target:.0%}  {'✓' if ok else '~'}")

print("\n▸ Reuse distribution across all used files:")
used = {f: c for f, c in reuse_count.items() if c > 0}
from collections import Counter
dist = Counter(used.values())
for times in sorted(dist):
    print(f"    used {times}×: {dist[times]} files")

print("\n" + "═" * 62)
print("  generation_plan.csv is ready for Phase 2 (mixing).")
print("═" * 62 + "\n")

# ─────────────────────────────────────────────────────────────────────────────
# 14. WRITE SHEET INTO pipeline_report.xlsx
# ─────────────────────────────────────────────────────────────────────────────
print("[Phase 1] Writing sheet to pipeline_report.xlsx ...")

def style_header_row(ws, fill_hex: str = "1F4E79"):
    fill   = PatternFill("solid", fgColor=fill_hex)
    font   = Font(bold=True, color="FFFFFF", size=9)
    align  = Alignment(horizontal="center", vertical="center", wrap_text=True)
    thin   = Side(style="thin", color="FFFFFF")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    for cell in ws[1]:
        cell.fill      = fill
        cell.font      = font
        cell.alignment = align
        cell.border    = border

def auto_col_width(ws, max_w: int = 40):
    for col in ws.columns:
        lengths = [len(str(c.value)) if c.value is not None else 0 for c in col]
        ws.column_dimensions[col[0].column_letter].width = min(max(lengths) + 2, max_w)

SHEET_NAME = "Generation Plan"
if REPORT_XLS.exists():
    wb = load_workbook(REPORT_XLS)
else:
    from openpyxl import Workbook
    wb = Workbook()
    if "Sheet" in wb.sheetnames:
        del wb["Sheet"]

if SHEET_NAME in wb.sheetnames:
    del wb[SHEET_NAME]
ws = wb.create_sheet(SHEET_NAME)

ws.append(list(df_out.columns))
for _, row in df_out.iterrows():
    ws.append([row[c] for c in df_out.columns])

style_header_row(ws)
auto_col_width(ws)
ws.freeze_panes = "A2"
ws.row_dimensions[1].height = 28

wb.save(REPORT_XLS)
print(f"  Saved sheet '{SHEET_NAME}' → {REPORT_XLS}\n")

print("[Phase 1 v3] COMPLETE ✓")
print(f"  Output: {PLAN_CSV}")
print(f"  Rows:   {len(df_out)}")
print(f"  Next:   Run phase2_mixing.py\n")

#PHASE 2


In [ ]:
"""
phase2_mixing.py
Samsung PRISM — Kitchen Audio Dataset v2
Phase 2: Audio Mixing

Reads generation_plan.csv, loads each stem with librosa at 32kHz,
applies gain, mixes all stems, peak-normalises, and saves to audio_32k/.
Also saves individual stems to stems/ for inspection.

Design rules (locked):
  - All audio: 32kHz, mono, 10 seconds (320000 samples)
  - Gain applied as: signal *= 10 ** (gain_dB / 20)
  - SNR formula: stem1_gain − stem3_gain = SNR_dB  (verified at save time)
  - Peak normalise final mix to −1 dBFS
  - status column in generation_plan.csv updated to "done" / "error"
  - Resumable: skips rows where status == "done"
  - Progress printed every 100 clips

Run on Google Colab CPU (slow but fine for 2500 clips).
GPU not needed — librosa + numpy mixing is CPU-only.
"""

# ─────────────────────────────────────────────────────────────────────────────
# 0. INSTALL & IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import os, sys, warnings, traceback
warnings.filterwarnings("ignore")

# Install soundfile if missing (needed for 32kHz WAV write)
try:
    import soundfile as sf
except ImportError:
    os.system("pip install soundfile -q")
    import soundfile as sf

import numpy as np
import pandas as pd
import librosa
from pathlib import Path

# ─────────────────────────────────────────────────────────────────────────────
# 1. MOUNT GOOGLE DRIVE
# ─────────────────────────────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
    print("[OK] Google Drive mounted.")
except ModuleNotFoundError:
    DRIVE_ROOT = os.path.expanduser("~/kitpri_dev")
    print(f"[INFO] Not in Colab — using local path: {DRIVE_ROOT}")

# ─────────────────────────────────────────────────────────────────────────────
# 2. PATHS
# ─────────────────────────────────────────────────────────────────────────────
BASE        = Path(DRIVE_ROOT) / "kitpri" / "kitpri_v2"
META_DIR    = BASE / "metadata"
AUDIO_DIR   = BASE / "audio_32k"
STEMS_DIR   = BASE / "stems"
PLAN_CSV    = META_DIR / "generation_plan.csv"

AUDIO_DIR.mkdir(parents=True, exist_ok=True)
STEMS_DIR.mkdir(parents=True, exist_ok=True)

assert PLAN_CSV.exists(), f"[ERROR] generation_plan.csv not found at {PLAN_CSV}"
print(f"[OK] generation_plan.csv found.")

# ─────────────────────────────────────────────────────────────────────────────
# 3. CONSTANTS
# ─────────────────────────────────────────────────────────────────────────────
SR          = 32000          # target sample rate
CLIP_LEN    = 10             # seconds
N_SAMPLES   = SR * CLIP_LEN  # 320000 samples
PEAK_TARGET = 10 ** (-1 / 20)  # −1 dBFS linear
SAVE_STEMS  = False          # set True to save individual stem WAVs (uses more Drive space)

# ─────────────────────────────────────────────────────────────────────────────
# 4. AUDIO HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def load_audio(path: str) -> np.ndarray:
    """
    Load a WAV/MP3/FLAC at 32kHz mono.
    Returns float32 array of exactly N_SAMPLES length (pad or trim).
    """
    y, _ = librosa.load(path, sr=SR, mono=True, dtype=np.float32)
    if len(y) >= N_SAMPLES:
        # Random crop for variety
        max_start = len(y) - N_SAMPLES
        start = np.random.randint(0, max_start + 1) if max_start > 0 else 0
        y = y[start : start + N_SAMPLES]
    else:
        # Loop-pad: repeat until long enough, then trim
        repeats = int(np.ceil(N_SAMPLES / len(y)))
        y = np.tile(y, repeats)[:N_SAMPLES]
    return y.astype(np.float32)


def apply_gain(y: np.ndarray, gain_dB: float) -> np.ndarray:
    """Apply gain in dB (linear scale)."""
    return y * (10 ** (gain_dB / 20.0))


def peak_normalise(y: np.ndarray, target_linear: float = PEAK_TARGET) -> np.ndarray:
    """Peak-normalise to target_linear. Skip if silent."""
    peak = np.max(np.abs(y))
    if peak < 1e-8:
        return y
    return y * (target_linear / peak)


def mix_stems(stems: list[np.ndarray]) -> np.ndarray:
    """Sum stems (all same length) into a single array."""
    mixed = np.zeros(N_SAMPLES, dtype=np.float32)
    for s in stems:
        mixed += s
    return mixed


def save_wav(y: np.ndarray, path: Path):
    """Save float32 array as 32kHz mono 16-bit WAV."""
    path.parent.mkdir(parents=True, exist_ok=True)
    # Clip to [-1, 1] before writing to avoid integer overflow
    y_clipped = np.clip(y, -1.0, 1.0)
    sf.write(str(path), y_clipped, SR, subtype="PCM_16")


# ─────────────────────────────────────────────────────────────────────────────
# 5. LOAD PLAN
# ─────────────────────────────────────────────────────────────────────────────
print("\n[Phase 2] Loading generation_plan.csv ...")
df = pd.read_csv(PLAN_CSV, dtype=str)
df.columns = [c.strip() for c in df.columns]

# Coerce numeric columns
for col in ["SNR_dB", "stem1_gain_dB", "stem2_gain_dB", "stem3_gain_dB", "stem4_gain_dB",
            "num_stems", "label"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

total      = len(df)
already_done = (df["status"] == "done").sum()
to_process   = total - already_done

print(f"  Total clips:    {total}")
print(f"  Already done:   {already_done}")
print(f"  To process:     {to_process}")

# Set numpy seed for reproducible random crops
np.random.seed(42)

# ─────────────────────────────────────────────────────────────────────────────
# 6. MAIN MIXING LOOP
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n[Phase 2] Mixing {to_process} clips ...")
print("  (This will take ~20–40 min on Colab CPU for 2500 clips)\n")

errors     = []
done_count = already_done

for idx, row in df.iterrows():
    if row["status"] == "done":
        continue

    file_id   = str(row["file_id"])
    label     = int(row["label"])
    num_stems = int(row["num_stems"])

    try:
        # ── Load stems ────────────────────────────────────────────────────
        stems_raw = []
        gain_dbs  = []

        for sn in range(1, num_stems + 1):
            fpath = str(row[f"stem{sn}_file"])
            gain  = float(row[f"stem{sn}_gain_dB"])
            y     = load_audio(fpath)
            stems_raw.append(y)
            gain_dbs.append(gain)

        # ── Apply gains ────────────────────────────────────────────────────
        stems_gained = [apply_gain(y, g) for y, g in zip(stems_raw, gain_dbs)]

        # ── Mix ────────────────────────────────────────────────────────────
        mixed = mix_stems(stems_gained)

        # ── Peak normalise ────────────────────────────────────────────────
        mixed = peak_normalise(mixed)

        # ── Save mixed clip ───────────────────────────────────────────────
        label_dir = "cooking" if label == 1 else "noncooking"
        out_path  = AUDIO_DIR / label_dir / f"{file_id}.wav"
        save_wav(mixed, out_path)

        # ── Optionally save stems ─────────────────────────────────────────
        if SAVE_STEMS:
            stem_dir = STEMS_DIR / label_dir / file_id
            for sn, (y_g, g) in enumerate(zip(stems_gained, gain_dbs), start=1):
                role = str(row[f"stem{sn}_role"])
                stem_path = stem_dir / f"stem{sn}_{role}.wav"
                save_wav(y_g, stem_path)

        # ── Mark done ─────────────────────────────────────────────────────
        df.at[idx, "status"] = "done"
        done_count += 1

        # ── Progress ──────────────────────────────────────────────────────
        if done_count % 100 == 0 or done_count == total:
            pct = 100 * done_count / total
            print(f"  [{done_count:>4}/{total}]  {pct:.1f}%  last: {file_id}")

            # Checkpoint: save updated CSV every 100 clips
            df.to_csv(PLAN_CSV, index=False)

    except Exception as e:
        err_msg = f"{file_id}: {type(e).__name__}: {e}"
        errors.append(err_msg)
        df.at[idx, "status"] = f"error: {type(e).__name__}"
        if len(errors) <= 5:
            print(f"  [ERROR] {err_msg}")
            traceback.print_exc()

# ─────────────────────────────────────────────────────────────────────────────
# 7. FINAL SAVE & REPORT
# ─────────────────────────────────────────────────────────────────────────────
df.to_csv(PLAN_CSV, index=False)
print(f"\n  Updated generation_plan.csv saved (status column updated).")

print("\n" + "═" * 60)
print("  PHASE 2 MIXING REPORT")
print("═" * 60)

status_counts = df["status"].value_counts()
n_done  = status_counts.get("done", 0)
n_error = total - n_done

print(f"\n  Total clips:    {total}")
print(f"  Done:           {n_done}")
print(f"  Errors:         {n_error}")

# Verify output files exist
cooking_files    = list((AUDIO_DIR / "cooking").glob("*.wav"))
noncooking_files = list((AUDIO_DIR / "noncooking").glob("*.wav"))
print(f"\n  WAV files on disk:")
print(f"    audio_32k/cooking/:    {len(cooking_files)}")
print(f"    audio_32k/noncooking/: {len(noncooking_files)}")
print(f"    Total:                 {len(cooking_files) + len(noncooking_files)}")

if errors:
    print(f"\n  First {min(10, len(errors))} errors:")
    for e in errors[:10]:
        print(f"    {e}")

# Spot-check: verify one clip's SNR
print("\n  SNR spot-check (first 3 clips):")
for _, row in df[df["status"] == "done"].head(3).iterrows():
    g1   = float(row["stem1_gain_dB"])
    g3   = float(row["stem3_gain_dB"])
    snr  = float(row["SNR_dB"])
    computed = round(g1 - g3, 2)
    ok   = abs(computed - snr) < 0.01
    print(f"    {row['file_id']}: g1={g1}, g3={g3}, "
          f"SNR_dB={snr}, computed={computed}  {'✓' if ok else '✗'}")

print("\n" + "═" * 60)
print("  audio_32k/ is ready for Phase 3 (RMS computation).")
print("═" * 60 + "\n")

print("[Phase 2] COMPLETE ✓")
print(f"  Output: {AUDIO_DIR}")
print(f"  Next:   Run phase3_rms.py\n")

#PHASE 3


In [ ]:
"""
phase3_rms.py
Samsung PRISM — Kitchen Audio Dataset v2
Phase 3: RMS Computation

Reads every WAV in audio_32k/, computes RMS (dBFS) and peak (dBFS),
and writes master_metadata.csv — the single source of truth for all
downstream phases (augmentation, training, evaluation).

master_metadata.csv columns:
  file_id, label, split, num_stems, SNR_dB, snr_band,
  mixing_method, status,
  stem1_class … stem4_gain_dB  (carried over from generation_plan),
  wav_path, rms_dBFS, peak_dBFS, duration_s, sr

Run on Google Colab CPU (~2–3 min for 2500 clips).
"""

# ─────────────────────────────────────────────────────────────────────────────
# 0. IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import os, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import soundfile as sf
from pathlib import Path

# ─────────────────────────────────────────────────────────────────────────────
# 1. MOUNT GOOGLE DRIVE
# ─────────────────────────────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
    print("[OK] Google Drive mounted.")
except ModuleNotFoundError:
    DRIVE_ROOT = os.path.expanduser("~/kitpri_dev")
    print(f"[INFO] Not in Colab — using local path: {DRIVE_ROOT}")

# ─────────────────────────────────────────────────────────────────────────────
# 2. PATHS
# ─────────────────────────────────────────────────────────────────────────────
BASE        = Path(DRIVE_ROOT) / "kitpri" / "kitpri_v2"
META_DIR    = BASE / "metadata"
AUDIO_DIR   = BASE / "audio_32k"
PLAN_CSV    = META_DIR / "generation_plan.csv"
MASTER_CSV  = META_DIR / "master_metadata.csv"

assert PLAN_CSV.exists(),  f"[ERROR] generation_plan.csv not found at {PLAN_CSV}"
assert AUDIO_DIR.exists(), f"[ERROR] audio_32k/ not found at {AUDIO_DIR}"
print(f"[OK] Paths verified.")

# ─────────────────────────────────────────────────────────────────────────────
# 3. HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def compute_rms_peak(path: str) -> tuple[float, float, float, int]:
    """
    Returns (rms_dBFS, peak_dBFS, duration_s, sample_rate).
    Uses soundfile for fast reading without resampling.
    """
    data, sr = sf.read(path, dtype="float32", always_2d=False)
    if data.ndim > 1:
        data = data.mean(axis=1)  # ensure mono

    rms_linear  = np.sqrt(np.mean(data ** 2))
    peak_linear = np.max(np.abs(data))

    rms_dBFS  = 20 * np.log10(rms_linear  + 1e-10)
    peak_dBFS = 20 * np.log10(peak_linear + 1e-10)
    duration  = len(data) / sr

    return round(rms_dBFS, 3), round(peak_dBFS, 3), round(duration, 4), sr

# ─────────────────────────────────────────────────────────────────────────────
# 4. LOAD GENERATION PLAN
# ─────────────────────────────────────────────────────────────────────────────
print("\n[Phase 3] Loading generation_plan.csv ...")
plan = pd.read_csv(PLAN_CSV)
plan.columns = [c.strip() for c in plan.columns]

# Keep only done clips
plan = plan[plan["status"] == "done"].reset_index(drop=True)
print(f"  Done clips in plan: {len(plan)}")

# ─────────────────────────────────────────────────────────────────────────────
# 5. COMPUTE RMS FOR EVERY WAV
# ─────────────────────────────────────────────────────────────────────────────
print("\n[Phase 3] Computing RMS / peak for all clips ...")

rms_rows = []
errors   = []

for idx, row in plan.iterrows():
    file_id   = str(row["file_id"])
    label     = int(row["label"])
    label_dir = "cooking" if label == 1 else "noncooking"
    wav_path  = AUDIO_DIR / label_dir / f"{file_id}.wav"

    if not wav_path.exists():
        errors.append(f"{file_id}: WAV not found at {wav_path}")
        continue

    try:
        rms, peak, dur, sr = compute_rms_peak(str(wav_path))
        rms_rows.append({
            "file_id":    file_id,
            "wav_path":   str(wav_path),
            "rms_dBFS":   rms,
            "peak_dBFS":  peak,
            "duration_s": dur,
            "sr":         sr,
        })
    except Exception as e:
        errors.append(f"{file_id}: {type(e).__name__}: {e}")

    if (idx + 1) % 500 == 0:
        print(f"  [{idx+1:>4}/{len(plan)}]  {100*(idx+1)/len(plan):.0f}%")

print(f"  [{len(plan)}/{len(plan)}]  100%")

rms_df = pd.DataFrame(rms_rows)
print(f"\n  RMS computed for {len(rms_df)} clips  ({len(errors)} errors)")

# ─────────────────────────────────────────────────────────────────────────────
# 6. MERGE INTO master_metadata.csv
# ─────────────────────────────────────────────────────────────────────────────
print("\n[Phase 3] Building master_metadata.csv ...")

master = plan.merge(rms_df, on="file_id", how="left")

# Compute snr_band from SNR_dB if not already present
if "snr_band" not in master.columns:
    def snr_to_band(snr):
        if   snr <  0: return "very_hard"
        elif snr <  5: return "hard"
        elif snr < 10: return "medium"
        elif snr < 20: return "easy"
        else:          return "very_easy"
    master["snr_band"] = pd.to_numeric(master["SNR_dB"], errors="coerce").apply(snr_to_band)

master.to_csv(MASTER_CSV, index=False)
print(f"  Saved: {MASTER_CSV}  ({len(master)} rows, {len(master.columns)} columns)")

# ─────────────────────────────────────────────────────────────────────────────
# 7. VALIDATION REPORT
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "═" * 58)
print("  PHASE 3 VALIDATION REPORT")
print("═" * 58)

print(f"\n▸ Clip count: {len(master)}  {'✓' if len(master)==2500 else '✗ (expected 2500)'}")

print("\n▸ RMS distribution (dBFS):")
for lbl, name in [(1, "cooking"), (0, "noncooking")]:
    sub = master[master["label"] == lbl]["rms_dBFS"].dropna()
    print(f"    {name:<12}  mean={sub.mean():.1f}  "
          f"std={sub.std():.1f}  "
          f"min={sub.min():.1f}  "
          f"max={sub.max():.1f}  dBFS")

print("\n▸ Peak distribution (dBFS):")
over_0  = (master["peak_dBFS"] > 0).sum()
over_m1 = (master["peak_dBFS"] > -1).sum()
print(f"    Clips with peak > 0 dBFS  (clipped):    {over_0}")
print(f"    Clips with peak > -1 dBFS (near limit): {over_m1}")
print(f"    Max peak seen: {master['peak_dBFS'].max():.2f} dBFS  "
      f"{'✓' if master['peak_dBFS'].max() <= -0.9 else '~'}")

print("\n▸ Duration check:")
wrong_dur = master[master["duration_s"].round(1) != 10.0]
print(f"    Clips not ~10.0s: {len(wrong_dur)}  "
      f"{'✓' if len(wrong_dur)==0 else '✗'}")

print("\n▸ Sample rate check:")
wrong_sr = master[master["sr"] != 32000]
print(f"    Clips not at 32kHz: {len(wrong_sr)}  "
      f"{'✓' if len(wrong_sr)==0 else '✗'}")

print("\n▸ Missing WAVs / errors:")
missing = master["rms_dBFS"].isna().sum()
print(f"    Missing: {missing}  {'✓' if missing==0 else '✗'}")
if errors:
    print(f"    First {min(5, len(errors))} errors:")
    for e in errors[:5]:
        print(f"      {e}")

print("\n▸ Split × label breakdown:")
pivot = master.groupby(["split", "label"]).size().unstack(fill_value=0)
print(pivot.to_string())

print("\n" + "═" * 58)
print("  master_metadata.csv is ready for Phase 4 (augmentation).")
print("═" * 58 + "\n")

print("[Phase 3] COMPLETE ✓")
print(f"  Output: {MASTER_CSV}")
print(f"  Next:   Run phase4_augmentation.py\n")

#PHASE 4


In [ ]:
"""
phase4_augmentation.py
Samsung PRISM — Kitchen Audio Dataset v2
Phase 4: Augmentation  (2,500 → 5,000 clips)

For each original clip in audio_32k/, generates exactly one augmented copy:
  - Odd-indexed clips  → pitch shift  (±1–3 semitones, randomly)
  - Even-indexed clips → time stretch (rate 0.85–1.15, randomly)

Augmented clips:
  - Same label and split as source
  - file_id prefixed with "aug_"  (e.g. aug_c_00001)
  - Saved to audio_32k/cooking/ and audio_32k/noncooking/ alongside originals
  - Appended to master_metadata.csv with aug_type column

Design notes:
  - Pitch shift uses librosa.effects.pitch_shift  (n_steps ∈ [-3, -2, -1, 1, 2, 3])
  - Time stretch uses librosa.effects.time_stretch (rate ∈ [0.85, 1.15])
    → after stretch, clip is trimmed or loop-padded back to exactly 10s
  - Peak-normalised to −1 dBFS after augmentation
  - Seed: 42 (reproducible)
  - Resumable: skips aug_ clips already present in master_metadata.csv

Run on Google Colab CPU (~25–45 min for 2500 augmented clips).
"""

# ─────────────────────────────────────────────────────────────────────────────
# 0. IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import os, warnings, traceback
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from pathlib import Path

# ─────────────────────────────────────────────────────────────────────────────
# 1. MOUNT GOOGLE DRIVE
# ─────────────────────────────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
    print("[OK] Google Drive mounted.")
except ModuleNotFoundError:
    DRIVE_ROOT = os.path.expanduser("~/kitpri_dev")
    print(f"[INFO] Not in Colab — using local path: {DRIVE_ROOT}")

# ─────────────────────────────────────────────────────────────────────────────
# 2. PATHS
# ─────────────────────────────────────────────────────────────────────────────
BASE        = Path(DRIVE_ROOT) / "kitpri" / "kitpri_v2"
META_DIR    = BASE / "metadata"
AUDIO_DIR   = BASE / "audio_32k"
MASTER_CSV  = META_DIR / "master_metadata.csv"

assert MASTER_CSV.exists(), f"[ERROR] master_metadata.csv not found at {MASTER_CSV}"
print(f"[OK] master_metadata.csv found.")

# ─────────────────────────────────────────────────────────────────────────────
# 3. CONSTANTS
# ─────────────────────────────────────────────────────────────────────────────
SR          = 32000
N_SAMPLES   = SR * 10          # 320000
PEAK_TARGET = 10 ** (-1 / 20)  # −1 dBFS linear
SEED        = 42

# Pitch shift: semitone choices (exclude 0)
PITCH_STEPS = [-3, -2, -1, 1, 2, 3]

# Time stretch: rate range
STRETCH_LOW, STRETCH_HIGH = 0.85, 1.15

# ─────────────────────────────────────────────────────────────────────────────
# 4. HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def load_wav(path: str) -> np.ndarray:
    """Load a 32kHz mono WAV, return float32 array of exactly N_SAMPLES."""
    data, _ = sf.read(path, dtype="float32", always_2d=False)
    if data.ndim > 1:
        data = data.mean(axis=1)
    return data[:N_SAMPLES] if len(data) >= N_SAMPLES else np.tile(data, int(np.ceil(N_SAMPLES / len(data))))[:N_SAMPLES]


def fix_length(y: np.ndarray) -> np.ndarray:
    """Trim or loop-pad to exactly N_SAMPLES."""
    if len(y) >= N_SAMPLES:
        return y[:N_SAMPLES]
    repeats = int(np.ceil(N_SAMPLES / len(y)))
    return np.tile(y, repeats)[:N_SAMPLES]


def peak_normalise(y: np.ndarray) -> np.ndarray:
    peak = np.max(np.abs(y))
    if peak < 1e-8:
        return y
    return y * (PEAK_TARGET / peak)


def save_wav(y: np.ndarray, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(path), np.clip(y, -1.0, 1.0), SR, subtype="PCM_16")


def compute_rms_peak(y: np.ndarray) -> tuple[float, float]:
    rms  = 20 * np.log10(np.sqrt(np.mean(y ** 2)) + 1e-10)
    peak = 20 * np.log10(np.max(np.abs(y))         + 1e-10)
    return round(rms, 3), round(peak, 3)


# ─────────────────────────────────────────────────────────────────────────────
# 5. LOAD MASTER METADATA — FIND ALREADY-DONE AUG CLIPS
# ─────────────────────────────────────────────────────────────────────────────
print("\n[Phase 4] Loading master_metadata.csv ...")
master = pd.read_csv(MASTER_CSV)
master.columns = [c.strip() for c in master.columns]

# Original clips only (no aug_ prefix)
originals = master[~master["file_id"].str.startswith("aug_")].copy()
originals = originals.reset_index(drop=True)

# Already-augmented file_ids
done_aug_ids = set(master[master["file_id"].str.startswith("aug_")]["file_id"].tolist())

print(f"  Original clips:   {len(originals)}")
print(f"  Already augmented: {len(done_aug_ids)}")
print(f"  To augment:        {len(originals) - len(done_aug_ids)}")

# ─────────────────────────────────────────────────────────────────────────────
# 6. AUGMENTATION LOOP
# ─────────────────────────────────────────────────────────────────────────────
rng = np.random.default_rng(SEED)
new_rows = []
errors   = []
done_count = len(done_aug_ids)

print(f"\n[Phase 4] Augmenting {len(originals) - len(done_aug_ids)} clips ...")
print("  (pitch shift on odd-index, time stretch on even-index)\n")

for i, row in originals.iterrows():
    file_id   = str(row["file_id"])
    aug_id    = f"aug_{file_id}"

    if aug_id in done_aug_ids:
        continue

    label     = int(row["label"])
    label_dir = "cooking" if label == 1 else "noncooking"
    src_path  = AUDIO_DIR / label_dir / f"{file_id}.wav"
    out_path  = AUDIO_DIR / label_dir / f"{aug_id}.wav"

    if not src_path.exists():
        errors.append(f"{file_id}: source WAV not found")
        continue

    try:
        y = load_wav(str(src_path))

        # Alternate: odd index → pitch shift, even index → time stretch
        if i % 2 == 1:
            # ── Pitch shift ──────────────────────────────────────────────
            n_steps = int(rng.choice(PITCH_STEPS))
            y_aug   = librosa.effects.pitch_shift(y, sr=SR, n_steps=n_steps)
            aug_type   = "pitch_shift"
            aug_param  = float(n_steps)
        else:
            # ── Time stretch ─────────────────────────────────────────────
            rate    = float(rng.uniform(STRETCH_LOW, STRETCH_HIGH))
            y_aug   = librosa.effects.time_stretch(y, rate=rate)
            y_aug   = fix_length(y_aug)
            aug_type   = "time_stretch"
            aug_param  = round(rate, 4)

        y_aug = fix_length(y_aug)
        y_aug = peak_normalise(y_aug)

        save_wav(y_aug, out_path)

        rms, peak = compute_rms_peak(y_aug)

        new_rows.append({
            **{c: row[c] for c in master.columns if c in row.index and c not in
               ["file_id", "wav_path", "rms_dBFS", "peak_dBFS", "aug_type", "aug_param"]},
            "file_id":    aug_id,
            "wav_path":   str(out_path),
            "rms_dBFS":   rms,
            "peak_dBFS":  peak,
            "duration_s": round(N_SAMPLES / SR, 4),
            "sr":         SR,
            "aug_type":   aug_type,
            "aug_param":  aug_param,
        })

        done_count += 1

        if done_count % 100 == 0 or done_count == len(originals):
            pct = 100 * done_count / len(originals)
            print(f"  [{done_count:>4}/{len(originals)}]  {pct:.1f}%  last: {aug_id}  ({aug_type}, param={aug_param})")

            # Checkpoint: append new rows and save
            if new_rows:
                aug_df = pd.DataFrame(new_rows)
                # Ensure aug_type / aug_param columns exist in master too
                for col in ["aug_type", "aug_param"]:
                    if col not in master.columns:
                        master[col] = ""
                master = pd.concat([master, aug_df], ignore_index=True)
                master.to_csv(MASTER_CSV, index=False)
                new_rows = []

    except Exception as e:
        err_msg = f"{file_id}: {type(e).__name__}: {e}"
        errors.append(err_msg)
        if len(errors) <= 5:
            print(f"  [ERROR] {err_msg}")

# Final save of any remaining rows
if new_rows:
    aug_df = pd.DataFrame(new_rows)
    for col in ["aug_type", "aug_param"]:
        if col not in master.columns:
            master[col] = ""
    master = pd.concat([master, aug_df], ignore_index=True)
    master.to_csv(MASTER_CSV, index=False)

# ─────────────────────────────────────────────────────────────────────────────
# 7. RELOAD AND VALIDATE
# ─────────────────────────────────────────────────────────────────────────────
master = pd.read_csv(MASTER_CSV)

print(f"\n  master_metadata.csv updated — {len(master)} total rows.")

print("\n" + "═" * 58)
print("  PHASE 4 VALIDATION REPORT")
print("═" * 58)

orig  = master[~master["file_id"].str.startswith("aug_")]
aug   = master[master["file_id"].str.startswith("aug_")]

print(f"\n▸ Total clips: {len(master)}  {'✓' if len(master)==5000 else f'✗ (expected 5000, got {len(master)})'}")
print(f"  Original:    {len(orig)}   {'✓' if len(orig)==2500 else '✗'}")
print(f"  Augmented:   {len(aug)}   {'✓' if len(aug)==2500 else '✗'}")

print("\n▸ Augmented clips by type:")
type_counts = aug["aug_type"].value_counts()
for t, c in type_counts.items():
    print(f"    {t:<15}: {c}")

print("\n▸ Label balance (augmented):")
aug_lbl = aug["label"].value_counts().sort_index()
for lbl, cnt in aug_lbl.items():
    name = "cooking" if lbl == 1 else "noncooking"
    print(f"    {name}: {cnt}  {'✓' if cnt==1250 else '✗'}")

print("\n▸ Split distribution (augmented + original combined):")
split_lbl = master.groupby(["split", "label"]).size().unstack(fill_value=0)
print(split_lbl.to_string())

print("\n▸ Peak check (augmented clips):")
over_0 = (aug["peak_dBFS"] > 0).sum()
print(f"    Clips with peak > 0 dBFS: {over_0}  {'✓' if over_0==0 else '✗'}")
print(f"    Max peak: {aug['peak_dBFS'].max():.2f} dBFS")

print("\n▸ WAV files on disk:")
for ldir in ["cooking", "noncooking"]:
    wavs = list((AUDIO_DIR / ldir).glob("*.wav"))
    orig_n = len([w for w in wavs if not w.stem.startswith("aug_")])
    aug_n  = len([w for w in wavs if w.stem.startswith("aug_")])
    print(f"    {ldir:<12}: {orig_n} original + {aug_n} augmented = {orig_n+aug_n} total")

if errors:
    print(f"\n▸ Errors: {len(errors)}")
    for e in errors[:5]:
        print(f"    {e}")
else:
    print(f"\n▸ Errors: 0  ✓")

print("\n" + "═" * 58)
print("  master_metadata.csv is ready for Phase 5 (CSV export).")
print("═" * 58 + "\n")

print("[Phase 4] COMPLETE ✓")
print(f"  Output: {MASTER_CSV}  ({len(master)} rows)")
print(f"  Next:   Run phase5_csv_export.py\n")

# PHASE 5 -> the full WAV existence check

In [ ]:
"""
phase5_csv_export.py
Samsung PRISM — Kitchen Audio Dataset v2
Phase 5: Train / Val / Test CSV Export

Reads master_metadata.csv, splits into train.csv / val.csv / test.csv.
Each CSV has exactly the columns needed by CNN14 and AST training scripts:

  file_path   — absolute path to the WAV file (32kHz mono, 10s)
  label       — 0 (non-cooking) or 1 (cooking)
  file_id     — unique clip identifier
  split       — train / val / test
  aug_type    — original / pitch_shift / time_stretch

Also writes a combined dataset.csv (all 5000 rows) for convenience,
and prints a final summary table.

Run on Google Colab CPU (fast — no audio processing).
"""

# ─────────────────────────────────────────────────────────────────────────────
# 0. IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import os, warnings
warnings.filterwarnings("ignore")

import pandas as pd
from pathlib import Path

# ─────────────────────────────────────────────────────────────────────────────
# 1. MOUNT GOOGLE DRIVE
# ─────────────────────────────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
    print("[OK] Google Drive mounted.")
except ModuleNotFoundError:
    DRIVE_ROOT = os.path.expanduser("~/kitpri_dev")
    print(f"[INFO] Not in Colab — using local path: {DRIVE_ROOT}")

# ─────────────────────────────────────────────────────────────────────────────
# 2. PATHS
# ─────────────────────────────────────────────────────────────────────────────
BASE       = Path(DRIVE_ROOT) / "kitpri" / "kitpri_v2"
META_DIR   = BASE / "metadata"
AUDIO_DIR  = BASE / "audio_32k"
MASTER_CSV = META_DIR / "master_metadata.csv"

assert MASTER_CSV.exists(), f"[ERROR] master_metadata.csv not found at {MASTER_CSV}"
print(f"[OK] master_metadata.csv found.")

# ─────────────────────────────────────────────────────────────────────────────
# 3. LOAD & PREPARE
# ─────────────────────────────────────────────────────────────────────────────
print("\n[Phase 5] Loading master_metadata.csv ...")
master = pd.read_csv(MASTER_CSV)
master.columns = [c.strip() for c in master.columns]
print(f"  Rows loaded: {len(master)}")

# Normalise aug_type: original clips have no aug_type entry
master["aug_type"] = master["aug_type"].fillna("original").replace("", "original")

# Ensure label is int
master["label"] = master["label"].astype(int)

# Build the minimal export columns (what training scripts need)
EXPORT_COLS = ["file_path", "label", "file_id", "split", "aug_type"]

# Verify wav_path column is present
assert "wav_path" in master.columns, "[ERROR] 'wav_path' column missing from master_metadata.csv"

# Rename wav_path → file_path for training script convention
export = master.rename(columns={"wav_path": "file_path"})[EXPORT_COLS].copy()

# Verify all WAV files exist (spot-check first + last 5, full check optional)
print("\n[Phase 5] Spot-checking WAV paths (first + last 5) ...")
check_rows = pd.concat([export.head(5), export.tail(5)])
missing = []
for _, row in check_rows.iterrows():
    if not Path(row["file_path"]).exists():
        missing.append(row["file_path"])

if missing:
    print(f"  [WARNING] {len(missing)} spot-check paths not found:")
    for p in missing:
        print(f"    {p}")
else:
    print("  ✓ All spot-check paths verified.")

# ─────────────────────────────────────────────────────────────────────────────
# 4. FULL WAV EXISTENCE CHECK (optional — set FULL_CHECK = True to enable)
# ─────────────────────────────────────────────────────────────────────────────
FULL_CHECK = True

if FULL_CHECK:
    print("\n[Phase 5] Full WAV existence check ...")
    missing_full = [row["file_path"] for _, row in export.iterrows()
                    if not Path(row["file_path"]).exists()]
    if missing_full:
        print(f"  [WARNING] {len(missing_full)} WAV files not found on disk.")
        for p in missing_full[:5]:
            print(f"    {p}")
    else:
        print(f"  ✓ All {len(export)} WAV paths verified on disk.")

# ─────────────────────────────────────────────────────────────────────────────
# 5. SPLIT AND SAVE
# ─────────────────────────────────────────────────────────────────────────────
print("\n[Phase 5] Writing split CSVs ...")

splits = {}
for split_name in ["train", "val", "test"]:
    df_split = export[export["split"] == split_name].copy()
    # Shuffle rows within each split (good practice before training)
    df_split = df_split.sample(frac=1, random_state=42).reset_index(drop=True)
    out_path = META_DIR / f"{split_name}.csv"
    df_split.to_csv(out_path, index=False)
    splits[split_name] = df_split
    print(f"  Saved {split_name}.csv  — {len(df_split)} rows  →  {out_path}")

# Also write combined dataset.csv
dataset_path = META_DIR / "dataset.csv"
export_shuffled = export.sample(frac=1, random_state=42).reset_index(drop=True)
export_shuffled.to_csv(dataset_path, index=False)
print(f"  Saved dataset.csv  — {len(export_shuffled)} rows  →  {dataset_path}")

# ─────────────────────────────────────────────────────────────────────────────
# 6. VALIDATION REPORT
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "═" * 62)
print("  PHASE 5 VALIDATION REPORT")
print("═" * 62)

print(f"\n▸ Total clips: {len(export)}  {'✓' if len(export)==5000 else '✗'}")

print("\n▸ Per-split breakdown:")
print(f"  {'split':<8} {'total':>7} {'cooking':>9} {'noncooking':>12} {'original':>10} {'augmented':>11}")
print("  " + "─" * 60)
for name, df_s in splits.items():
    total   = len(df_s)
    cooking = (df_s["label"] == 1).sum()
    nonc    = (df_s["label"] == 0).sum()
    orig    = (df_s["aug_type"] == "original").sum()
    aug     = (df_s["aug_type"] != "original").sum()
    bal_ok  = "✓" if abs(cooking - nonc) <= 4 else "~"
    print(f"  {name:<8} {total:>7} {cooking:>9} {nonc:>12} {orig:>10} {aug:>11}  {bal_ok}")

print("\n▸ Augmentation type breakdown (train only):")
train_aug = splits["train"]["aug_type"].value_counts()
for t, c in train_aug.items():
    print(f"    {t:<15}: {c}")

print("\n▸ Label distribution check:")
for name, df_s in splits.items():
    cooking = (df_s["label"] == 1).sum()
    nonc    = (df_s["label"] == 0).sum()
    ratio   = cooking / len(df_s)
    ok      = 0.48 <= ratio <= 0.52
    print(f"    {name:<6}: cooking={cooking}  noncooking={nonc}  ratio={ratio:.3f}  {'✓' if ok else '~'}")

print("\n▸ CSV column check:")
for name, df_s in splits.items():
    cols_ok = set(EXPORT_COLS).issubset(set(df_s.columns))
    print(f"    {name}.csv columns: {list(df_s.columns)}  {'✓' if cols_ok else '✗'}")

print("\n▸ Sample rows from train.csv:")
print(splits["train"][["file_id", "label", "aug_type", "file_path"]].head(4).to_string(index=False))

print("\n" + "═" * 62)
print("  train.csv / val.csv / test.csv are ready for model training.")
print("═" * 62 + "\n")

print("[Phase 5] COMPLETE ✓")
print(f"  Outputs:")
print(f"    {META_DIR}/train.csv   ({len(splits['train'])} rows)")
print(f"    {META_DIR}/val.csv     ({len(splits['val'])} rows)")
print(f"    {META_DIR}/test.csv    ({len(splits['test'])} rows)")
print(f"    {META_DIR}/dataset.csv ({len(export)} rows)")
print(f"\n  Next:   Upload dataset to Kaggle (Phase 6) then run phase7_training.py\n")

# AUDIT DRIVE -> PRE PHASE 6

In [ ]:
"""
drive_audit.py
Samsung PRISM — Kitchen Audio Dataset v2
Complete Drive audit — run this anytime to verify dataset integrity.
"""

import os, warnings
warnings.filterwarnings("ignore")

import pandas as pd
from pathlib import Path

# ── Mount ──────────────────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except ModuleNotFoundError:
    DRIVE_ROOT = os.path.expanduser("~/kitpri_dev")

BASE      = Path(DRIVE_ROOT) / "kitpri" / "kitpri_v2"
META_DIR  = BASE / "metadata"
AUDIO_DIR = BASE / "audio_32k"
RAW_DIR   = Path(DRIVE_ROOT) / "kitpri" / "raw_sources"

print("═" * 65)
print("  KITPRI_V2 — COMPLETE DRIVE AUDIT")
print("═" * 65)

# ─────────────────────────────────────────────────────────────────────────────
# 1. FOLDER STRUCTURE
# ─────────────────────────────────────────────────────────────────────────────
print("\n▸ TOP-LEVEL FOLDER STRUCTURE:")
for folder in [BASE, META_DIR, AUDIO_DIR, RAW_DIR]:
    exists = "✓ EXISTS" if folder.exists() else "✗ MISSING"
    print(f"    {exists}  {folder}")

# ─────────────────────────────────────────────────────────────────────────────
# 2. METADATA FILES
# ─────────────────────────────────────────────────────────────────────────────
print("\n▸ METADATA FILES:")
meta_files = [
    "inventory.csv",
    "generation_plan.csv",
    "master_metadata.csv",
    "train.csv",
    "val.csv",
    "test.csv",
    "dataset.csv",
    "pipeline_report.xlsx",
    "coverage_summary.csv",
]

meta_summary = {}
for fname in meta_files:
    fpath = META_DIR / fname
    if fpath.exists():
        size_kb = fpath.stat().st_size / 1024
        if fname.endswith(".csv"):
            try:
                df = pd.read_csv(fpath)
                meta_summary[fname] = df
                print(f"    ✓  {fname:<30} {len(df):>6} rows × {len(df.columns):>2} cols   {size_kb:>8.1f} KB")
            except Exception as e:
                print(f"    ✓  {fname:<30} (unreadable: {e})   {size_kb:.1f} KB")
        else:
            print(f"    ✓  {fname:<30} {size_kb:>8.1f} KB")
    else:
        print(f"    ✗  {fname:<30} MISSING")

# ─────────────────────────────────────────────────────────────────────────────
# 3. AUDIO FILES
# ─────────────────────────────────────────────────────────────────────────────
print("\n▸ AUDIO FILES (audio_32k/):")
total_wav = 0
for ldir in ["cooking", "noncooking"]:
    lpath = AUDIO_DIR / ldir
    if lpath.exists():
        wavs      = list(lpath.glob("*.wav"))
        originals = [w for w in wavs if not w.stem.startswith("aug_")]
        augmented = [w for w in wavs if w.stem.startswith("aug_")]
        total_wav += len(wavs)
        size_mb = sum(w.stat().st_size for w in wavs) / (1024**2)
        print(f"    ✓  audio_32k/{ldir}/")
        print(f"         original : {len(originals):>5} WAVs")
        print(f"         augmented: {len(augmented):>5} WAVs")
        print(f"         total    : {len(wavs):>5} WAVs   {size_mb:.1f} MB")
    else:
        print(f"    ✗  audio_32k/{ldir}/  MISSING")

print(f"\n    TOTAL WAVs on disk: {total_wav}  {'✓' if total_wav==5000 else f'✗ (expected 5000)'}")

# ─────────────────────────────────────────────────────────────────────────────
# 4. CSV CONTENT VERIFICATION
# ─────────────────────────────────────────────────────────────────────────────
print("\n▸ CSV CONTENT VERIFICATION:")

# train / val / test
expected = {"train": 3500, "val": 750, "test": 750, "dataset": 5000}
for name, exp_rows in expected.items():
    fname = f"{name}.csv"
    if fname not in meta_summary:
        print(f"    ✗  {fname} — not loaded")
        continue
    df = meta_summary[fname]
    row_ok  = "✓" if len(df) == exp_rows else f"✗ (expected {exp_rows})"
    col_ok  = "✓" if {"file_path","label","file_id","split","aug_type"}.issubset(df.columns) else "✗ missing cols"
    bal     = ""
    if "label" in df.columns:
        c = (df["label"]==1).sum()
        n = (df["label"]==0).sum()
        bal = f"  cooking={c} noncooking={n}"
    print(f"    {fname:<16} rows={len(df)} {row_ok}   cols {col_ok}{bal}")

# master_metadata
if "master_metadata.csv" in meta_summary:
    mm = meta_summary["master_metadata.csv"]
    print(f"\n    master_metadata.csv:")
    print(f"      rows     : {len(mm)}  {'✓' if len(mm)==5000 else '✗'}")
    print(f"      columns  : {list(mm.columns)}")
    if "split" in mm.columns:
        print(f"      splits   : {mm['split'].value_counts().to_dict()}")
    if "label" in mm.columns:
        print(f"      labels   : {mm['label'].value_counts().to_dict()}")
    if "aug_type" in mm.columns:
        mm["aug_type"] = mm["aug_type"].fillna("original").replace("","original")
        print(f"      aug_type : {mm['aug_type'].value_counts().to_dict()}")
    if "rms_dBFS" in mm.columns:
        print(f"      rms_dBFS : mean={mm['rms_dBFS'].mean():.1f}  std={mm['rms_dBFS'].std():.1f}  "
              f"min={mm['rms_dBFS'].min():.1f}  max={mm['rms_dBFS'].max():.1f}")

# ─────────────────────────────────────────────────────────────────────────────
# 5. WAV PATH CROSS-CHECK (CSV ↔ DISK)
# ─────────────────────────────────────────────────────────────────────────────
print("\n▸ WAV PATH CROSS-CHECK (train/val/test CSVs ↔ disk):")
all_missing = []
for name in ["train", "val", "test"]:
    fname = f"{name}.csv"
    if fname not in meta_summary:
        continue
    df = meta_summary[fname]
    if "file_path" not in df.columns:
        print(f"    ✗  {fname}: no file_path column")
        continue
    missing = [p for p in df["file_path"] if not Path(p).exists()]
    all_missing.extend(missing)
    status = "✓" if not missing else f"✗ {len(missing)} missing"
    print(f"    {fname:<16} {status}")

if all_missing:
    print(f"\n    First 5 missing paths:")
    for p in all_missing[:5]:
        print(f"      {p}")
else:
    print(f"\n    ✓ All WAV paths in train/val/test CSVs verified on disk.")

# ─────────────────────────────────────────────────────────────────────────────
# 6. RAW SOURCES SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print("\n▸ RAW SOURCES (raw_sources/):")
if RAW_DIR.exists():
    fg_dir = RAW_DIR / "foreground"
    bg_dir = RAW_DIR / "background"
    for category, cdir in [("foreground", fg_dir), ("background", bg_dir)]:
        if cdir.exists():
            classes = [d for d in cdir.iterdir() if d.is_dir()]
            total_files = sum(len(list(d.glob("*.*"))) for d in classes)
            print(f"    {category}/  — {len(classes)} classes, {total_files} files")
            for cls in sorted(classes):
                n = len(list(cls.glob("*.*")))
                print(f"      {cls.name:<22}: {n:>4} files")
        else:
            print(f"    ✗  {category}/ MISSING")
else:
    print(f"    ✗  raw_sources/ not found at {RAW_DIR}")

# ─────────────────────────────────────────────────────────────────────────────
# 7. FINAL SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "═" * 65)
print("  AUDIT SUMMARY")
print("═" * 65)

checks = {
    "master_metadata.csv (5000 rows)": "master_metadata.csv" in meta_summary and len(meta_summary["master_metadata.csv"]) == 5000,
    "train.csv (3500 rows)":           "train.csv" in meta_summary and len(meta_summary["train.csv"]) == 3500,
    "val.csv (750 rows)":              "val.csv" in meta_summary and len(meta_summary["val.csv"]) == 750,
    "test.csv (750 rows)":             "test.csv" in meta_summary and len(meta_summary["test.csv"]) == 750,
    "5000 WAVs on disk":               total_wav == 5000,
    "All WAV paths in CSVs valid":     len(all_missing) == 0,
    "generation_plan.csv exists":      "generation_plan.csv" in meta_summary,
}

all_ok = True
for desc, ok in checks.items():
    print(f"  {'✓' if ok else '✗'}  {desc}")
    if not ok:
        all_ok = False

print()
if all_ok:
    print("  ✅ DATASET IS COMPLETE AND CONSISTENT. Ready for Phase 6 (Kaggle upload).")
else:
    print("  ⚠️  Some checks failed — review items marked ✗ above.")
print("═" * 65 + "\n")

In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/kitpri")

for item in sorted(BASE.rglob("*")):
    depth = len(item.relative_to(BASE).parts) - 1
    prefix = "    " * depth + "├── "
    if item.is_dir():
        n = len(list(item.iterdir()))
        print(f"{prefix}{item.name}/  ({n} items)")
    else:
        size_kb = item.stat().st_size / 1024
        print(f"{prefix}{item.name}  ({size_kb:.1f} KB)")

In [ ]:
from pathlib import Path
base = Path("/content/drive/MyDrive/kitpri/kitpri_v2")
print("kitpri_v2 exists:", base.exists())
for item in sorted(base.iterdir()):
    n = len(list(item.rglob("*"))) if item.is_dir() else "-"
    print(f"  {item.name}/  ({n} items)")

In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/kitpri")

def show_tree(path, depth=0, max_depth=4):
    if depth > max_depth:
        return
    indent = "    " * depth + "├── "
    if path.is_dir():
        n_files = len([f for f in path.iterdir() if f.is_file()])
        n_dirs  = len([f for f in path.iterdir() if f.is_dir()])
        print(f"{indent}{path.name}/  📁 [{n_dirs} folders, {n_files} files]")
        for child in sorted(path.iterdir()):
            if child.is_dir():
                show_tree(child, depth + 1, max_depth)

show_tree(BASE)

# PHASE 6

In [ ]:
"""
Phase 6 — Step 1: Fix absolute Colab paths → relative paths in all CSVs
Run in Google Colab (CPU runtime)

Before: /content/drive/MyDrive/kitpri/kitpri_v2/audio_32k/cooking/c_00001.wav
After:  audio_32k/cooking/c_00001.wav

Training scripts will prepend DATA_ROOT at runtime.
"""

import pandas as pd
import os

# ── CONFIG ────────────────────────────────────────────────────────────────────
DRIVE_BASE = "/content/drive/MyDrive/kitpri/kitpri_v2"
META_DIR   = f"{DRIVE_BASE}/metadata"

CSV_FILES = [
    f"{META_DIR}/train.csv",
    f"{META_DIR}/val.csv",
    f"{META_DIR}/test.csv",
    f"{META_DIR}/dataset.csv",
]

# The absolute prefix to strip from file_path values
ABS_PREFIX = "/content/drive/MyDrive/kitpri/kitpri_v2/"

# ── HELPER ────────────────────────────────────────────────────────────────────
def fix_paths(csv_path: str) -> dict:
    """Convert absolute paths to relative paths in a single CSV. Returns stats."""
    df = pd.read_csv(csv_path)

    if "file_path" not in df.columns:
        return {"file": csv_path, "status": "SKIPPED — no file_path column"}

    original_sample = df["file_path"].iloc[0]

    # Count how many actually need fixing
    needs_fix = df["file_path"].str.startswith(ABS_PREFIX).sum()

    if needs_fix == 0:
        # Check if paths look already relative
        already_relative = df["file_path"].str.startswith("audio_32k/").sum()
        return {
            "file": os.path.basename(csv_path),
            "rows": len(df),
            "fixed": 0,
            "status": f"ALREADY RELATIVE ({already_relative}/{len(df)} start with audio_32k/)"
        }

    # Strip the absolute prefix
    df["file_path"] = df["file_path"].str.replace(ABS_PREFIX, "", regex=False)

    # Verify no absolute paths remain
    still_absolute = df["file_path"].str.startswith("/").sum()
    if still_absolute > 0:
        return {
            "file": os.path.basename(csv_path),
            "status": f"ERROR — {still_absolute} paths still absolute after fix!"
        }

    # Write back in-place
    df.to_csv(csv_path, index=False)

    return {
        "file": os.path.basename(csv_path),
        "rows": len(df),
        "fixed": int(needs_fix),
        "sample_before": original_sample,
        "sample_after": df["file_path"].iloc[0],
        "status": "OK ✓"
    }

# ── MAIN ──────────────────────────────────────────────────────────────────────
def main():
    print("=" * 60)
    print("Phase 6 — Path Fix: absolute → relative")
    print("=" * 60)

    all_ok = True

    for csv_path in CSV_FILES:
        if not os.path.exists(csv_path):
            print(f"\n⚠️  NOT FOUND: {csv_path}")
            all_ok = False
            continue

        result = fix_paths(csv_path)
        print(f"\n📄 {result['file']}")
        print(f"   Status : {result['status']}")
        if "rows" in result:
            print(f"   Rows   : {result['rows']:,}")
        if "fixed" in result and result["fixed"] > 0:
            print(f"   Fixed  : {result['fixed']:,} paths")
            print(f"   Before : {result.get('sample_before', 'n/a')}")
            print(f"   After  : {result.get('sample_after', 'n/a')}")
        if "ERROR" in result["status"]:
            all_ok = False

    print("\n" + "=" * 60)

    # ── VERIFICATION PASS ─────────────────────────────────────────────────────
    print("\n🔍 Verification pass:")
    errors = 0
    for csv_path in CSV_FILES:
        if not os.path.exists(csv_path):
            continue
        df = pd.read_csv(csv_path)
        if "file_path" not in df.columns:
            continue
        abs_count = df["file_path"].str.startswith("/").sum()
        rel_count = df["file_path"].str.startswith("audio_32k/").sum()
        ok = abs_count == 0 and rel_count == len(df)
        status = "✅ PASS" if ok else "❌ FAIL"
        print(f"   {status}  {os.path.basename(csv_path)}: "
              f"{rel_count}/{len(df)} relative, {abs_count} absolute")
        if not ok:
            errors += 1

    print("\n" + "=" * 60)
    if errors == 0 and all_ok:
        print("✅ All CSVs have clean relative paths. Ready for Phase 6 Step 2.")
    else:
        print("❌ Some files have issues — review output above before proceeding.")
    print("=" * 60)


if __name__ == "__main__":
    main()

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!kaggle datasets create \
    -p /content/drive/MyDrive/kitpri/kitpri_v2 \
    --dir-mode zip \
    --public

In [ ]:
"""
Phase 6 — Step 2: Upload kitpri_v2 to Kaggle
Run in Google Colab (CPU runtime), AFTER phase6_fix_paths.py passes.

Uploads:
  kitpri_v2/audio_32k/   — 5000 WAVs
  kitpri_v2/metadata/    — CSVs + pipeline report

Kaggle dataset slug: ayushalia/kitpri-v2
"""

import os
import json
import subprocess

# ── CONFIG ────────────────────────────────────────────────────────────────────
DATASET_TITLE = "kitpri-v2"
DATASET_ID    = "ayushalia/kitpri-v2"

DRIVE_BASE = "/content/drive/MyDrive/kitpri/kitpri_v2"

# ── STEP 0 — Mount Drive ──────────────────────────────────────────────────────
print("Step 0 — Mounting Google Drive...")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("  Drive mounted ✓")
except Exception as e:
    print(f"  ⚠️  Drive mount: {e} (may already be mounted)")

# ── STEP 1 — Install Kaggle CLI ───────────────────────────────────────────────
print("\nStep 1 — Installing Kaggle CLI...")
subprocess.run(["pip", "install", "kaggle", "-q"], check=True)
print("  kaggle installed ✓")

# ── STEP 2 — Set API credentials ─────────────────────────────────────────────
print("\nStep 2 — Using Kaggle credentials from ~/.kaggle/kaggle.json ✓")

# ── STEP 3 — Verify paths are relative (safety check) ────────────────────────
print("\nStep 3 — Verifying CSV paths are relative...")
import pandas as pd

meta_dir = os.path.join(DRIVE_BASE, "metadata")
for csv_name in ["train.csv", "val.csv", "test.csv", "dataset.csv"]:
    csv_path = os.path.join(meta_dir, csv_name)
    if not os.path.exists(csv_path):
        print(f"  ⚠️  {csv_name} not found — skipping check")
        continue
    df = pd.read_csv(csv_path)
    if "file_path" in df.columns:
        abs_count = df["file_path"].str.startswith("/").sum()
        if abs_count > 0:
            raise RuntimeError(
                f"❌ {csv_name} still has {abs_count} absolute paths! "
                f"Run phase6_fix_paths.py first."
            )
        print(f"  ✅ {csv_name}: all paths relative")
    else:
        print(f"  ⚠️  {csv_name}: no file_path column")

# ── STEP 4 — Write dataset-metadata.json ─────────────────────────────────────
print("\nStep 4 — Writing dataset-metadata.json...")
metadata = {
    "title": DATASET_TITLE,
    "id": DATASET_ID,
    "licenses": [{"name": "CC0-1.0"}],
    "subtitle": "Kitchen audio classification dataset — 5000 clips, 32kHz mono 10s WAV",
    "description": (
        "Binary kitchen audio classification dataset (cooking vs non-cooking). "
        "5000 clips: 2500 cooking / 2500 non-cooking. "
        "32kHz mono 10s WAV, peak -1 dBFS. "
        "Train/val/test splits (3500/750/750) with augmentation (pitch_shift, time_stretch). "
        "Samsung PRISM project — CNN14 + AST ensemble."
    ),
    "keywords": ["audio", "classification", "kitchen", "cooking", "deep learning"]
}

metadata_path = os.path.join(DRIVE_BASE, "dataset-metadata.json")
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)
print(f"  Written to {metadata_path} ✓")

# ── STEP 5 — Confirm directory structure ─────────────────────────────────────
print("\nStep 5 — Confirming kitpri_v2 structure...")

audio_cooking    = os.path.join(DRIVE_BASE, "audio_32k", "cooking")
audio_noncooking = os.path.join(DRIVE_BASE, "audio_32k", "noncooking")

cooking_count    = len([f for f in os.listdir(audio_cooking)    if f.endswith(".wav")]) if os.path.isdir(audio_cooking)    else 0
noncooking_count = len([f for f in os.listdir(audio_noncooking) if f.endswith(".wav")]) if os.path.isdir(audio_noncooking) else 0

print(f"  audio_32k/cooking/    : {cooking_count:,} WAVs")
print(f"  audio_32k/noncooking/ : {noncooking_count:,} WAVs")
print(f"  Total                 : {cooking_count + noncooking_count:,} WAVs")

if cooking_count + noncooking_count != 5000:
    print(f"  ⚠️  Expected 5000 WAVs, found {cooking_count + noncooking_count}")
else:
    print("  ✅ 5000 WAVs confirmed")

# ── STEP 6 — Upload to Kaggle ─────────────────────────────────────────────────
print("\nStep 6 — Uploading to Kaggle (this will take a while — ~800MB)...")
print("  Do NOT close this tab while uploading.\n")

result = subprocess.run(
    ["kaggle", "datasets", "create", "-p", DRIVE_BASE, "--dir-mode", "zip"],
    capture_output=True,
    text=True
)

print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    print("\n❌ Upload failed. Common fixes:")
    print("   • 401 Unauthorized → token expired, regenerate at kaggle.com/settings/account")
    print("   • Dataset already exists → use 'kaggle datasets version' instead of 'create'")
    print("   • Folder not found → check DRIVE_BASE path is correct")
else:
    print(f"\n✅ Upload complete!")
    print(f"   Dataset URL: https://www.kaggle.com/datasets/{DATASET_ID}")
    print(f"\n   In your Kaggle training notebook:")
    print(f"   1. Click + Add Data → search '{DATASET_TITLE}'")
    print(f"   2. Set DATA_ROOT = '/kaggle/input/{DATASET_TITLE}'")
    print(f"   3. Paths in CSVs are relative, so: DATA_ROOT / file_path resolves correctly")

# ── STEP 7 — (Optional) Version bump if dataset already exists ────────────────
print("\n" + "─" * 50)
print("NOTE: If Step 6 failed with 'dataset already exists', run this instead:")
print(f"""
  !kaggle datasets version -p {DRIVE_BASE} \\
      --dir-mode zip \\
      -m "kitpri_v2: corrected labels, relative paths"
""")

In [ ]:
!kaggle datasets list -s audio

# mix check

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/kitpri/kitpri_v2/metadata/master_metadata.csv")
print(df.columns.tolist())
print(df[["foreground_file", "background_file", "snr_db"]].head(10))

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/kitpri/kitpri_v2/metadata/master_metadata.csv")

# Check stem files and SNR
print(df[["file_id", "num_stems", "SNR_dB", "snr_band",
          "stem1_file", "stem1_role",
          "stem2_file", "stem2_role",
          "stem3_file", "stem3_role",
          "mixing_method", "status"]].head(10).to_string())


In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/kitpri/kitpri_v2/metadata/master_metadata.csv")

# Stem count distribution overall
print("=== Stem count distribution ===")
print(df["num_stems"].value_counts().sort_index())

print("\n=== Stem count by label ===")
print(df.groupby(["label", "num_stems"]).size().unstack(fill_value=0))

print("\n=== Stem count by split ===")
print(df.groupby(["split", "num_stems"]).size().unstack(fill_value=0))

print("\n=== Stem count by label + split ===")
print(df.groupby(["label", "split", "num_stems"]).size().unstack(fill_value=0))

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/kitpri/kitpri_v2/metadata/master_metadata.csv")

# Look at gain levels for each stem role
print("=== stem1 (fg_primary) gain_dB stats ===")
print(df["stem1_gain_dB"].describe())

print("\n=== stem2 (fg_secondary) gain_dB stats ===")
print(df["stem2_gain_dB"].describe())

print("\n=== stem3 (bg_primary) gain_dB stats ===")
print(df["stem3_gain_dB"].describe())

print("\n=== stem4 gain_dB stats ===")
print(df["stem4_gain_dB"].describe())

# Find clips where gains are most balanced (fg and bg closest in level)
df["fg_bg_diff"] = abs(df["stem1_gain_dB"] - df["stem3_gain_dB"])
most_balanced = df.nsmallest(10, "fg_bg_diff")[["file_id", "label", "SNR_dB", "snr_band",
                                                  "stem1_gain_dB", "stem2_gain_dB",
                                                  "stem3_gain_dB", "fg_bg_diff", "wav_path"]]
print("\n=== 10 most balanced fg/bg clips ===")
print(most_balanced.to_string())

In [ ]:
import IPython.display as ipd

# Most balanced — fg and bg almost equal
ipd.display(ipd.Audio("/content/drive/MyDrive/kitpri/kitpri_v2/audio_32k/noncooking/n_00453.wav"))

# Balanced cooking clip
ipd.display(ipd.Audio("/content/drive/MyDrive/kitpri/kitpri_v2/audio_32k/cooking/c_00553.wav"))

# Very hard SNR (negative SNR — bg actually louder than fg)
ipd.display(ipd.Audio("/content/drive/MyDrive/kitpri/kitpri_v2/audio_32k/noncooking/n_01024.wav"))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!zip -r /content/kitpri_v2.zip /content/drive/MyDrive/kitpri/kitpri_v2/
print("Zip done ✓")

In [ ]:
from google.colab import files
files.download('/content/kitpri_v2.zip')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!zip -r /content/kitpri_v2.zip /content/drive/MyDrive/kitpri/kitpri_v2/
print("Done")

In [ ]:
# Install Kaggle CLI locally
!pip install kaggle

# Place your kaggle.json in C:\Users\<you>\.kaggle\kaggle.json
# Then upload
kaggle datasets create -p "path\to\kitpri_v2_folder" --dir-mode zip